# TCC 2: Reconstrução 3D de fumaça e segmentação 3D sem ground truth

Autor: Felipe Augusto Sousa de Almada (UFMA, Engenharia da Computação)

Pipeline para o Google Colab com GPU. Os dados e resultados são gravados em `/content/drive/MyDrive/TCC2_FluidNexus`. Execute as células na ordem. O passo a passo está em `docs/PLANO_DE_DESENVOLVIMENTO.md`.

## 1. Dados do FluidNexus e código oficial

In [ ]:
#@title Criar estrutura do projeto e listar os datasets
from google.colab import drive

drive.mount("/content/drive")
from pathlib import Path
import pandas as pd

RAIZ = Path("/content/drive/MyDrive/TCC2_FluidNexus")
PASTAS = {
    nome: RAIZ / caminho
    for nome, caminho in {
        "compactados": "dados/compactados",
        "extraidos": "dados/extraidos",
        "codigo": "codigo",
        "modelos": "modelos",
        "resultados": "resultados",
        "figuras": "figuras",
        "logs": "logs",
    }.items()
}
[pasta.mkdir(parents=True, exist_ok=True) for pasta in PASTAS.values()]
datasets = pd.DataFrame(
    [
        {
            "dataset": "FluidNexus-Smoke",
            "arquivo": "FluidNexusSmoke.zip",
            "tamanho_compactado": "2,53 GB",
            "cenas_publicacao": 120,
        },
        {
            "dataset": "FluidNexus-Ball",
            "arquivo": "FluidNexusBall.zip",
            "tamanho_compactado": "3,17 GB",
            "cenas_publicacao": 120,
        },
        {
            "dataset": "ScalarReal",
            "arquivo": "ScalarReal.zip",
            "tamanho_compactado": "78,1 MB",
            "cenas_publicacao": "1 amostra disponibilizada",
        },
    ]
)
display(datasets)
print(f"Raiz do projeto: {RAIZ}")
print("Pastas preparadas:", ", ".join(PASTAS))


In [ ]:
#@title Baixar e inspecionar o arquivo ScalarReal
import urllib.request, zipfile
from pathlib import PurePosixPath

url = "https://huggingface.co/datasets/yuegao/FluidNexusDatasets/resolve/main/ScalarReal.zip?download=true"
arquivo = PASTAS["compactados"] / "ScalarReal.zip"
urllib.request.urlretrieve(url, arquivo) if not arquivo.exists() else None
zip_scalar = zipfile.ZipFile(arquivo)
membros = zip_scalar.infolist()
inseguros = [
    m.filename
    for m in membros
    if PurePosixPath(m.filename.replace("\\", "/")).is_absolute()
    or ".." in PurePosixPath(m.filename.replace("\\", "/")).parts
]
amostra_zip = pd.DataFrame(
    [
        {
            "caminho": m.filename,
            "compactado_kb": round(m.compress_size / 1024, 1),
            "original_kb": round(m.file_size / 1024, 1),
        }
        for m in membros[:20]
    ]
)
display(amostra_zip)
print(f"Arquivo: {arquivo}")
print(
    f"Tamanho baixado: {arquivo.stat().st_size/1024**2:.1f} MB | Itens: {len(membros)} | Caminhos inseguros: {len(inseguros)}"
)
zip_scalar.close()


In [ ]:
#@title Extrair e contabilizar as imagens do ScalarReal
import zipfile
from collections import Counter

destino_extracao = PASTAS["extraidos"]
raiz_scalar = destino_extracao / "ScalarReal"
zipfile.ZipFile(arquivo).extractall(destino_extracao) if not raiz_scalar.exists() else None
contagem = Counter(p.parent.name for p in raiz_scalar.rglob("*.png"))
tabela_contagem = pd.DataFrame(sorted(contagem.items()), columns=["sequencia", "quantidade_png"])
display(tabela_contagem)
print(f"Pasta extraída: {raiz_scalar}")
print(f"Total de imagens PNG: {sum(contagem.values())}")


In [ ]:
#@title Baixar e verificar o arquivo FluidNexus-Smoke
from huggingface_hub import hf_hub_download
import zipfile
from pathlib import Path, PurePosixPath

caminho_smoke = Path(
    hf_hub_download(
        repo_id="yuegao/FluidNexusDatasets",
        repo_type="dataset",
        filename="FluidNexusSmoke.zip",
        local_dir=str(PASTAS["compactados"]),
    )
)
zip_smoke = zipfile.ZipFile(caminho_smoke)
membros_smoke = zip_smoke.infolist()
inseguros_smoke = [
    m.filename
    for m in membros_smoke
    if PurePosixPath(m.filename.replace("\\", "/")).is_absolute()
    or ".." in PurePosixPath(m.filename.replace("\\", "/")).parts
]
amostra_smoke = pd.DataFrame(
    [
        {
            "caminho": m.filename,
            "compactado_mb": round(m.compress_size / 1024**2, 2),
            "original_mb": round(m.file_size / 1024**2, 2),
        }
        for m in membros_smoke[:20]
    ]
)
display(amostra_smoke)
print(f"Arquivo: {caminho_smoke}")
print(
    f"Tamanho baixado: {caminho_smoke.stat().st_size/1024**3:.2f} GB | Itens: {len(membros_smoke)} | Caminhos inseguros: {len(inseguros_smoke)}"
)
zip_smoke.close()


In [ ]:
#@title Extrair e contabilizar as imagens do FluidNexus-Smoke
import zipfile
from collections import Counter

raiz_smoke = PASTAS["extraidos"] / "FluidNexusSmoke"
zipfile.ZipFile(caminho_smoke).extractall(PASTAS["extraidos"]) if not raiz_smoke.exists() else None
contagem_smoke = Counter(p.parent.name for p in raiz_smoke.rglob("*.png"))
tabela_smoke = pd.DataFrame(sorted(contagem_smoke.items()), columns=["camera", "quantidade_png"])
display(tabela_smoke)
print(f"Pasta extraída: {raiz_smoke}")
print(f"Total de imagens PNG: {sum(contagem_smoke.values())}")


In [ ]:
#@title Baixar o conjunto FluidNexus-Ball
from huggingface_hub import hf_hub_download

caminho_ball = Path(
    hf_hub_download(
        repo_id="yuegao/FluidNexusDatasets",
        filename="FluidNexusBall.zip",
        repo_type="dataset",
        local_dir=str(PASTAS["compactados"]),
    )
)
print(f"Arquivo baixado: {caminho_ball}")
print(f"Tamanho: {caminho_ball.stat().st_size/1024**3:.2f} GiB")


In [ ]:
#@title Extrair e contabilizar as imagens do FluidNexus-Ball
from collections import Counter

raiz_ball = PASTAS["extraidos"] / "FluidNexusBall"
zipfile.ZipFile(caminho_ball).extractall(PASTAS["extraidos"]) if not raiz_ball.exists() else None
contagem_ball = Counter(p.parent.name for p in raiz_ball.rglob("*.png"))
print("EXTRAÇÃO DO FLUIDNEXUS-BALL")
print(f"Pasta: {raiz_ball}")
print(f"Total PNG: {sum(contagem_ball.values())}")
[print(f"{camera}: {quantidade} imagens") for camera, quantidade in sorted(contagem_ball.items())]


In [ ]:
#@title Clonar e inventariar o repositório oficial FluidNexus
import subprocess

repo_fluidnexus = PASTAS["codigo"] / "FluidNexus"
resultado_clone = (
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ueoo/FluidNexus.git", str(repo_fluidnexus)],
        capture_output=True,
        text=True,
    )
    if not repo_fluidnexus.exists()
    else None
)
print("CÓDIGO OFICIAL FLUIDNEXUS")
print(
    resultado_clone.stdout
    if resultado_clone and resultado_clone.stdout
    else "Repositório já presente no Drive."
)
print(resultado_clone.stderr if resultado_clone and resultado_clone.stderr else "")
print(f"Pasta: {repo_fluidnexus}")
print("Arquivos principais:", sorted(p.name for p in repo_fluidnexus.iterdir())[:30])


## 2. Ambiente do FluidNexus

In [ ]:
#@title [Versão alta] Restaurar e recompilar o ambiente do FluidNexus (detecta a GPU: L4/A100/T4)
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path
import os
import subprocess
import sys
import time

ENV_DIR = Path("/content/fluidnexus_py312")
PYTHON_ENV = ENV_DIR / "bin" / "python"
PROJECT_ROOT = Path("/content/drive/MyDrive/TCC2_FluidNexus")
REPO_ROOT = PROJECT_ROOT / "codigo" / "FluidNexus"
SUBMODULES = REPO_ROOT / "FluidDynamics" / "submodules"
LOG_PATH = PROJECT_ROOT / "logs" / "restauracao_ambiente_versao_alta.txt"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_LOCAL = Path("/content/restauracao_ambiente_versao_alta.txt")

log_file = LOG_LOCAL.open("w", encoding="utf-8")


def emit(message=""):
    text = str(message)
    print(text, flush=True)
    log_file.write(text + "\n")
    log_file.flush()


def run(command, extra_env=None, cwd=None):
    emit()
    emit("$ " + " ".join(map(str, command)))
    process_env = os.environ.copy()
    if extra_env:
        process_env.update(extra_env)
    process = subprocess.Popen(
        [str(item) for item in command],
        cwd=str(cwd) if cwd else None,
        env=process_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        emit(line.rstrip())
    code = process.wait()
    emit(f"[codigo={code}]")
    if code != 0:
        raise RuntimeError(f"Comando falhou com código {code}: {command}")


try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
    ).strip()
    emit("=" * 72)
    emit("RESTAURAÇÃO COMPLETA DO AMBIENTE: TCC 2 / FLUIDNEXUS (VERSÃO ALTA)")
    emit("Autor: Felipe Augusto Sousa de Almada")
    emit(f"GPU: {gpu}")
    cap = (
        subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True)
        .strip()
        .splitlines()[0]
    )
    import psutil

    emit(f"Capacidade computacional: {cap}")
    emit(f"RAM total: {psutil.virtual_memory().total/1e9:.1f} GB")
    emit("=" * 72)

    if not PYTHON_ENV.exists():
        run(["uv", "venv", "--python", "3.12", str(ENV_DIR)])
    else:
        emit(f"Ambiente já existe e será atualizado: {ENV_DIR}")

    run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            str(PYTHON_ENV),
            "--index-url",
            "https://download.pytorch.org/whl/cu130",
            "torch==2.11.0",
            "torchvision==0.26.0",
        ]
    )

    run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            str(PYTHON_ENV),
            "--find-links",
            "https://data.pyg.org/whl/torch-2.11.0+cu130.html",
            "torch-cluster==1.6.3",
            "torch-scatter==2.1.2",
        ]
    )

    run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            str(PYTHON_ENV),
            "ninja",
            "setuptools",
            "wheel",
            "packaging",
            "pyyaml",
            "plyfile",
            "tqdm",
            "scipy",
            "imageio",
            "opencv-python-headless",
            "tensorboard",
            "lovely-numpy==0.2.13",
            "lovely-tensors==0.1.18",
            "kornia==0.8.0",
            "kornia-rs==0.1.8",
            "moviepy==1.0.3",
        ]
    )

    cuda_home = "/usr/local/cuda-13.0" if Path("/usr/local/cuda-13.0").exists() else "/usr/local/cuda"
    build_env = {
        "CUDA_HOME": cuda_home,
        "TORCH_CUDA_ARCH_LIST": cap,
        "MAX_JOBS": "4",
    }
    emit(f"CUDA_HOME selecionado: {cuda_home}")

    extensions = [
        ("simple-knn", SUBMODULES / "simple-knn"),
        ("gaussian-rasterization-ch1", SUBMODULES / "gaussian_rasterization_ch1"),
        ("gaussian-rasterization-ch3", SUBMODULES / "gaussian_rasterization_ch3"),
    ]
    for name, folder in extensions:
        if not folder.exists():
            raise FileNotFoundError(f"Submódulo não encontrado: {folder}")
        emit(f"\n=== Compilando {name} ===")
        import shutil as _sh

        for sub in ["build"] + [p.name for p in folder.glob("*.egg-info")]:
            if (folder / sub).exists():
                _sh.rmtree(folder / sub)
                emit(f"Removido cache antigo de compilação: {folder / sub}")
        run(
            ["uv", "pip", "install", "--python", str(PYTHON_ENV), "--no-build-isolation", str(folder)],
            extra_env=build_env,
        )

    site_packages = ENV_DIR / "lib" / "python3.12" / "site-packages"
    alias_dir = site_packages / "diff_gaussian_rasterization"
    alias_dir.mkdir(parents=True, exist_ok=True)
    alias_file = alias_dir / "__init__.py"
    alias_file.write_text(
        "# Alias de compatibilidade para o FluidNexus\n"
        "# O ScalarReal usa diretamente ch1; este alias atende somente o import genérico de renderer/__init__.py.\n"
        "from diff_gaussian_rasterization_ch3 import *\n",
        encoding="utf-8",
    )
    emit(f"Alias de compatibilidade criado em: {alias_file}")

    validation = """
import importlib.metadata as md
import torch
import torchvision
import torch_cluster
import torch_scatter
import simple_knn
import diff_gaussian_rasterization_ch1
import diff_gaussian_rasterization_ch3
import diff_gaussian_rasterization
import lovely_numpy
import lovely_tensors
import kornia
from moviepy.editor import VideoFileClip
print('Python OK')
print('moviepy: OK')
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('torch-cluster:', md.version('torch-cluster'))
print('torch-scatter:', md.version('torch-scatter'))
print('kornia:', kornia.__version__)
print('CUDA ativa:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('simple_knn: OK')
print('rasterizador ch1: OK')
print('rasterizador ch3: OK')
print('alias genérico: OK')
"""
    run([str(PYTHON_ENV), "-c", validation], extra_env=build_env)

    emit()
    emit("AMBIENTE COMPLETO RESTAURADO E VALIDADO COM SUCESSO")
    emit(f"Log salvo em: {LOG_PATH}")
finally:
    if not log_file.closed:
        log_file.close()
    import shutil

    try:
        os.listdir(PROJECT_ROOT)
    except OSError:
        drive.mount("/content/drive", force_remount=True)
    shutil.copy(LOG_LOCAL, LOG_PATH)


In [ ]:
#@title Corrigir compatibilidade do campo bbox_model no FluidNexus
from pathlib import Path
import shutil
import subprocess
from datetime import datetime, timezone

arquivo = Path("/content/drive/MyDrive/TCC2_FluidNexus/codigo/FluidNexus/FluidDynamics/scene/scene_info.py")
backup = arquivo.with_suffix(".py.pre_bbox_model.bak")
log = Path("/content/drive/MyDrive/TCC2_FluidNexus/logs/correcao_bbox_model_scalarreal.txt")
texto_antes = arquivo.read_text(encoding="utf-8")

if not backup.exists():
    shutil.copy2(arquivo, backup)

if "bbox_model:" not in texto_antes:
    alvo = "    ply_path: str"
    if alvo not in texto_antes:
        raise RuntimeError("Linha ply_path não encontrada; correção não aplicada.")
    texto_depois = texto_antes.replace(alvo, alvo + "\n    bbox_model: object = None", 1)
    arquivo.write_text(texto_depois, encoding="utf-8")
    status = "Campo bbox_model adicionado com valor padrão None."
else:
    texto_depois = texto_antes
    status = "Campo bbox_model já estava presente; nenhuma alteração adicional foi feita."

repo = arquivo.parents[1]
python_env = Path("/content/fluidnexus_py312/bin/python")
teste = """
from scene.scene_info import SceneInfo
objeto = SceneInfo(None, [], [], {}, None)
print('Campos:', SceneInfo._fields)
print('bbox_model padrão:', objeto.bbox_model)
assert 'bbox_model' in SceneInfo._fields
assert objeto.bbox_model is None
print('VALIDAÇÃO DA CORREÇÃO: SUCESSO')
"""
resultado = subprocess.run(
    [str(python_env), "-c", teste],
    cwd=repo,
    capture_output=True,
    text=True,
)

registro = "\n".join(
    [
        f"Data UTC: {datetime.now(timezone.utc).isoformat()}",
        "Projeto: TCC 2 - FluidNexus",
        "Autor: Felipe Augusto Sousa de Almada",
        f"Arquivo alterado: {arquivo}",
        f"Backup: {backup}",
        f"Status: {status}",
        f"Código da validação: {resultado.returncode}",
        resultado.stdout,
        resultado.stderr,
    ]
)
log.write_text(registro, encoding="utf-8")
print(registro)
if resultado.returncode != 0:
    raise RuntimeError("A estrutura SceneInfo não passou na validação.")


## 3. Reconstrução 3D: versão leve

In [ ]:
#@title Preparar dataset e configuração do piloto ScalarReal (frames 0 a 40)
from pathlib import Path
import json, shutil
from datetime import datetime, timezone

raiz = Path("/content/drive/MyDrive/TCC2_FluidNexus")
origem = raiz / "dados/extraidos/ScalarReal"
destino = raiz / "dados/processados/ScalarReal_piloto_0_40"
config_oficial = raiz / "codigo/FluidNexus/FluidDynamics/configs/scalar_real.json"
config_destino = raiz / "configuracoes/scalar_real_piloto_0_40.json"
modelo_destino = raiz / "modelos/ScalarReal_piloto_0_40/fisico"

for pasta in [destino, config_destino.parent, modelo_destino, raiz / "logs"]:
    pasta.mkdir(parents=True, exist_ok=True)

copias = 0
for frame in range(41):
    pasta_frame = destino / "colmap_frames" / f"colmap_{frame}" / "input"
    pasta_frame.mkdir(parents=True, exist_ok=True)
    for camera in range(5):
        fonte = origem / f"train{camera:02d}" / f"{frame}.png"
        alvo = pasta_frame / f"train{camera:02d}.png"
        if not fonte.exists():
            raise FileNotFoundError(f"Imagem ausente: {fonte}")
        if not alvo.exists() or alvo.stat().st_size != fonte.stat().st_size:
            shutil.copy2(fonte, alvo)
            copias += 1

base = json.loads((origem / "transforms.json").read_text(encoding="utf-8"))


def salvar_split(nome, cameras):
    conteudo = {chave: valor for chave, valor in base.items() if chave != "frames"}
    conteudo["frames"] = [f for f in base["frames"] if int(f["file_path"][-1]) in cameras]
    (destino / nome).write_text(json.dumps(conteudo, ensure_ascii=False, indent=2), encoding="utf-8")


salvar_split("transforms_train_scalar_real.json", {0, 1, 3, 4})
salvar_split("transforms_test_scalar_real.json", {2})
salvar_split("transforms_train_test_scalar_real.json", {0, 1, 2, 3, 4})
shutil.copy2(origem / "info.json", destino / "info.json")

config = json.loads(config_oficial.read_text(encoding="utf-8"))
config.update(
    {
        "resolution": 2,
        "start_time": 0,
        "duration": 41,
        "time_step": 1,
        "train_views": "0134",
        "train_views_fake": "",
        "test_views_fake": "",
        "use_refined_fake": False,
        "future_pred_frames": 0,
        "iterations_per_time_first": 200,
        "iterations_per_time_current": 50,
        "iterations_per_time_current_max": 50,
        "iterations_per_time_current_sparse": 50,
        "iterations_per_time_current_level_two": 50,
        "iterations_per_time_current_level_two_max": 50,
        "test_iteration": 2200,
        "save_iterations": [200, 500, 1000, 2200],
        "test_iterations": [200, 500, 1000, 2200],
        "spherical_cam_start": 0,
        "spherical_cam_duration": 41,
        "spherical_cam_step": 1,
    }
)
config_destino.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

manifesto = {
    "autor": "Felipe Augusto Sousa de Almada",
    "tipo": "piloto multivista direto; não é o experimento final",
    "frames": {"inicial": 0, "intermediarios": [10, 20], "final": 40, "total": 41},
    "cameras_treino": [0, 1, 3, 4],
    "camera_teste": 2,
    "usa_zero123_cogvideox": False,
    "resolucao": "metade da resolução original (resolution=2)",
    "configuracao": str(config_destino),
    "dados": str(destino),
    "saida_modelo_fisico": str(modelo_destino),
    "criado_utc": datetime.now(timezone.utc).isoformat(),
}
(destino / "manifesto_piloto.json").write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8"
)

esperados = 41 * 5
existentes = sum(1 for _ in (destino / "colmap_frames").glob("colmap_*/input/train*.png"))
print("Dataset piloto:", destino)
print(f"Imagens preparadas: {existentes}/{esperados} (novas cópias nesta execução: {copias})")
print("Configuração:", config_destino)
print("Saída planejada:", modelo_destino)
print("Frames do experimento: 0, 10, 20 e 40 (40 = final)")
assert existentes == esperados
assert json.loads((destino / "transforms_train_scalar_real.json").read_text())["frames"].__len__() == 4
assert json.loads((destino / "transforms_test_scalar_real.json").read_text())["frames"].__len__() == 1
print("Preparação concluída e validada.")


In [ ]:
#@title Preparar dados e treinar a reconstrução física do ScalarReal (frames 20 a 160, configuração leve)
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess

projeto = Path("/content/drive/MyDrive/TCC2_FluidNexus")
repo = projeto / "codigo" / "FluidNexus" / "FluidDynamics"
entrada = repo / "entries_scalar_real" / "train_physical_particle.py"
python_env = Path("/content/fluidnexus_py312/bin/python")
origem = projeto / "dados" / "extraidos" / "ScalarReal"
piloto = projeto / "dados" / "processados" / "ScalarReal_piloto_0_40"
dados = projeto / "dados" / "processados" / "ScalarReal_20_160"
modelo = projeto / "modelos" / "ScalarReal_20_160" / "fisico"
config_base = projeto / "configuracoes" / "scalar_real_piloto_0_40.json"
config_execucao = projeto / "configuracoes" / "scalar_real_20_160_fisico.json"
log = projeto / "logs" / "treinamento_fisico_scalarreal_20_160.txt"

INICIO, FIM = 20, 160
DURACAO = FIM - INICIO + 1

assert python_env.exists(), "Ambiente Python 3.12 ausente: rode a célula de restauração do ambiente."
if modelo.exists() and any(modelo.iterdir()):
    raise RuntimeError(f"Já existe um modelo em {modelo}. Nada foi sobrescrito.")
for pasta in [dados, modelo, log.parent]:
    pasta.mkdir(parents=True, exist_ok=True)

copias = 0
for frame in range(0, FIM + 1):
    pasta_frame = dados / "colmap_frames" / f"colmap_{frame}" / "input"
    pasta_frame.mkdir(parents=True, exist_ok=True)
    for camera in range(5):
        fonte = origem / f"train{camera:02d}" / f"{frame}.png"
        alvo = pasta_frame / f"train{camera:02d}.png"
        if not fonte.exists():
            raise FileNotFoundError(f"Imagem ausente: {fonte}")
        if not alvo.exists() or alvo.stat().st_size != fonte.stat().st_size:
            shutil.copy2(fonte, alvo)
            copias += 1
for nome in [
    "transforms_train_scalar_real.json",
    "transforms_test_scalar_real.json",
    "transforms_train_test_scalar_real.json",
    "info.json",
]:
    shutil.copy2(piloto / nome, dados / nome)
total = sum(1 for _ in (dados / "colmap_frames").glob("colmap_*/input/train*.png"))
print(f"Imagens preparadas: {total}/{(FIM + 1) * 5} (novas cópias: {copias})")
assert total == (FIM + 1) * 5

config = json.loads(config_base.read_text(encoding="utf-8"))
config.pop("test_iteration", None)
config.update(
    {
        "data_path": str(dados),
        "model_path": str(modelo),
        "loader": "scalar_real",
        "start_time": INICIO,
        "duration": DURACAO,
        "spherical_cam_start": INICIO,
        "spherical_cam_duration": DURACAO,
    }
)
config_execucao.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
(dados / "manifesto.json").write_text(
    json.dumps(
        {
            "autor": "Felipe Augusto Sousa de Almada",
            "tipo": "experimento ScalarReal, configuração leve (resolução 1/2, 200/50 iterações)",
            "frames_treino": [INICIO, FIM],
            "frames_figura": [40, 80, 120, 160],
            "cameras_treino": [0, 1, 3, 4],
            "camera_teste": 2,
            "criado_utc": datetime.now(timezone.utc).isoformat(),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

ambiente = os.environ.copy()
ambiente.update({"CUDA_HOME": "/usr/local/cuda-13.0", "TORCH_CUDA_ARCH_LIST": "7.5", "PYTHONUNBUFFERED": "1"})
comando = [
    str(python_env),
    "-u",
    str(entrada),
    "--data_path",
    str(dados),
    "--model_path",
    str(modelo),
    "--loader",
    "scalar_real",
    "--config_path",
    str(config_execucao),
]
cabecalho = "\n".join(
    [
        "=" * 78,
        f"TREINAMENTO FÍSICO SCALARREAL — FRAMES {INICIO} A {FIM} (LEVE)",
        "Autor: Felipe Augusto Sousa de Almada",
        f"Início UTC: {datetime.now(timezone.utc).isoformat()}",
        f"Dados: {dados}",
        f"Modelo: {modelo}",
        f"Configuração: {config_execucao}",
        "=" * 78,
    ]
)
print(cabecalho, flush=True)
with log.open("w", encoding="utf-8") as arq:
    arq.write(cabecalho + "\n")
    proc = subprocess.Popen(
        comando,
        cwd=repo,
        env=ambiente,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for linha in proc.stdout:
        print(linha, end="", flush=True)
        arq.write(linha)
        arq.flush()
    codigo = proc.wait()
    final = f"Código de saída: {codigo} | Fim UTC: {datetime.now(timezone.utc).isoformat()}"
    arq.write(final + "\n")
print(final)
if codigo != 0:
    raise RuntimeError(f"O treinamento físico falhou. Veja o log: {log}")
print("TREINAMENTO FÍSICO SCALARREAL 20-160 CONCLUÍDO COM SUCESSO")


In [ ]:
#@title Treinar a reconstrução visual do ScalarReal (frames 20 a 160, configuração leve)
from pathlib import Path
from datetime import datetime, timezone
import json, os, re, subprocess

projeto = Path("/content/drive/MyDrive/TCC2_FluidNexus")
repo = projeto / "codigo" / "FluidNexus" / "FluidDynamics"
entrada = repo / "entries_scalar_real" / "train_visual_particle.py"
python_env = Path("/content/fluidnexus_py312/bin/python")
dados = projeto / "dados" / "processados" / "ScalarReal_20_160"
modelo_fisico = projeto / "modelos" / "ScalarReal_20_160" / "fisico"
modelo_visual = projeto / "modelos" / "ScalarReal_20_160" / "visual"
config_fisico = projeto / "configuracoes" / "scalar_real_20_160_fisico.json"
config_visual = projeto / "configuracoes" / "scalar_real_20_160_visual.json"
log = projeto / "logs" / "treinamento_visual_scalarreal_20_160.txt"

assert python_env.exists(), "Ambiente Python 3.12 ausente: rode a célula de restauração do ambiente."
assert (modelo_fisico / "checkpoint").exists(), f"Checkpoints físicos não encontrados em {modelo_fisico}"
if modelo_visual.exists() and any(modelo_visual.iterdir()):
    raise RuntimeError(f"Já existe um modelo visual em {modelo_visual}. Nada foi sobrescrito.")
modelo_visual.mkdir(parents=True, exist_ok=True)

config = json.loads(config_fisico.read_text(encoding="utf-8"))
config.update({"data_path": str(dados), "model_path": str(modelo_visual), "load_path": str(modelo_fisico)})
config_visual.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
print("Intervalo:", config["start_time"], "a", config["start_time"] + config["duration"] - 1)

ambiente = os.environ.copy()
ambiente.update({"CUDA_HOME": "/usr/local/cuda-13.0", "TORCH_CUDA_ARCH_LIST": "7.5", "PYTHONUNBUFFERED": "1"})
comando = [
    str(python_env),
    "-u",
    str(entrada),
    "--data_path",
    str(dados),
    "--model_path",
    str(modelo_visual),
    "--load_path",
    str(modelo_fisico),
    "--loader",
    "scalar_real",
    "--config_path",
    str(config_visual),
]
cabecalho = "\n".join(
    [
        "=" * 78,
        "TREINAMENTO VISUAL SCALARREAL — FRAMES 20 A 160 (LEVE)",
        "Autor: Felipe Augusto Sousa de Almada",
        f"Início UTC: {datetime.now(timezone.utc).isoformat()}",
        f"Modelo físico: {modelo_fisico}",
        f"Saída visual: {modelo_visual}",
        f"Log completo: {log}",
        "=" * 78,
    ]
)
print(cabecalho, flush=True)

barra = re.compile(r"\d+%\|")
ultimo = None
with log.open("w", encoding="utf-8") as arq:
    arq.write(cabecalho + "\n")
    proc = subprocess.Popen(
        comando,
        cwd=repo,
        env=ambiente,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for linha in proc.stdout:
        arq.write(linha)
        arq.flush()
        if barra.search(linha):
            m = re.search(r"(\d+)/(\d+) \[", linha)
            if linha.lstrip().startswith(("Simulating", "Optimizing current", "Training")) and m:
                passo = int(m.group(1))
                if passo != ultimo and (passo % 10 == 0 or passo == int(m.group(2))):
                    ultimo = passo
                    print(f"  progresso geral: {passo}/{m.group(2)}", flush=True)
            continue
        if linha.strip():
            print(linha.rstrip()[:200], flush=True)
    codigo = proc.wait()
    final = f"Código de saída: {codigo} | Fim UTC: {datetime.now(timezone.utc).isoformat()}"
    arq.write(final + "\n")
print(final)
if codigo != 0:
    raise RuntimeError(f"O treinamento visual falhou. Veja o log: {log}")
print("TREINAMENTO VISUAL SCALARREAL 20-160 CONCLUÍDO COM SUCESSO")


In [ ]:
#@title [Smoke] Criar câmeras de treino/teste e validar fundo + físico (poucas iterações)
from pathlib import Path
from datetime import datetime, timezone
import json, os, re, shutil, subprocess

projeto = Path("/content/drive/MyDrive/TCC2_FluidNexus")
repo = projeto / "codigo" / "FluidNexus" / "FluidDynamics"
py = Path("/content/fluidnexus_py312/bin/python")
dados = projeto / "dados" / "extraidos" / "FluidNexusSmoke"
cfg_dir = projeto / "configuracoes"
modelos = projeto / "modelos" / "Smoke_teste_minimo"
assert py.exists(), "Ambiente Python 3.12 ausente: rode a célula de restauração."

base = json.loads((dados / "transforms.json").read_text())


def split(nome, cams):
    c = {k: v for k, v in base.items() if k != "frames"}
    c["frames"] = [f for f in base["frames"] if f["file_path"][-1] in cams]
    (dados / nome).write_text(json.dumps(c, indent=2))
    return [f["file_path"] for f in c["frames"]]


print("Treino:", split("transforms_train.json", "0134"))
print("Teste :", split("transforms_test.json", "2"))


def cfg(oficial, nome, mudancas):
    c = json.loads((repo / "configs" / oficial).read_text())
    c.pop("test_iteration", None)
    c.update(mudancas)
    p = cfg_dir / nome
    p.write_text(json.dumps(c, indent=2))
    return p


comum = {
    "resolution": 2,
    "train_views": "0134",
    "train_views_fake": "",
    "test_views_fake": "",
    "use_refined_fake": False,
}
cfg_bg = cfg(
    "fluid_nexus_smoke_background.json",
    "smoke_teste_fundo.json",
    {**comum, "iterations": 30, "save_iterations": [30], "test_iterations": [30]},
)
cfg_fis = cfg(
    "fluid_nexus_smoke_dynamics.json",
    "smoke_teste_fisico.json",
    {
        **comum,
        "start_time": 40,
        "duration": 2,
        "time_step": 1,
        "bg_load_iteration": 30,
        "iterations_per_time_first": 2,
        "iterations_per_time_current": 1,
        "iterations_per_time_current_max": 1,
        "iterations_per_time_current_sparse": 1,
        "iterations_per_time_current_level_two": 1,
        "iterations_per_time_current_level_two_max": 1,
        "save_iterations": [2],
        "test_iterations": [2],
        "spherical_cam_start": 40,
        "spherical_cam_duration": 2,
    },
)

ambiente = os.environ.copy()
ambiente.update({"CUDA_HOME": "/usr/local/cuda-13.0", "TORCH_CUDA_ARCH_LIST": "7.5", "PYTHONUNBUFFERED": "1"})
barra = re.compile(r"\d+%\|")


def rodar(nome, args):
    log_local = Path(f"/content/{nome}.txt")
    print("\n" + "=" * 70 + f"\n{nome}\n" + "=" * 70, flush=True)
    with log_local.open("w") as arq:
        p = subprocess.Popen(
            [str(py), "-u"] + [str(a) for a in args],
            cwd=repo,
            env=ambiente,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for linha in p.stdout:
            arq.write(linha)
            if not barra.search(linha) and linha.strip():
                print(linha.rstrip()[:220], flush=True)
        codigo = p.wait()
    try:
        shutil.copy2(log_local, projeto / "logs" / log_local.name)
    except Exception as e:
        print("Aviso: log não copiado para o Drive:", e)
    print(f">>> {nome}: código de saída {codigo}")
    return codigo


if modelos.exists():
    shutil.rmtree(modelos)
c1 = rodar(
    "smoke_teste_fundo",
    [
        "entries_fluid_nexus/train_background.py",
        "--data_path",
        dados,
        "--config_path",
        cfg_bg,
        "--loader",
        "fluid_nexus_real",
        "--model_path",
        modelos / "fundo",
    ],
)
c2 = None
if c1 == 0:
    c2 = rodar(
        "smoke_teste_fisico",
        [
            "entries_fluid_nexus/train_physical_particle.py",
            "--loader",
            "fluid_nexus_real",
            "--data_path",
            dados,
            "--config_path",
            cfg_fis,
            "--bg_load_path",
            modelos / "fundo",
            "--model_path",
            modelos / "fisico",
        ],
    )
print(
    "\nRESUMO: fundo =",
    "OK" if c1 == 0 else "FALHOU",
    "| físico =",
    "OK" if c2 == 0 else ("não rodou" if c2 is None else "FALHOU"),
)


In [ ]:
#@title [Smoke] Treinar fundo + reconstrução física (frames 20 a 160, configuração leve)
from pathlib import Path
from datetime import datetime, timezone
import json, os, re, shutil, subprocess, time

projeto = Path("/content/drive/MyDrive/TCC2_FluidNexus")
repo = projeto / "codigo" / "FluidNexus" / "FluidDynamics"
py = Path("/content/fluidnexus_py312/bin/python")
dados = projeto / "dados" / "extraidos" / "FluidNexusSmoke"
cfg_dir = projeto / "configuracoes"
modelos = projeto / "modelos" / "Smoke_20_160"
INICIO, FIM = 20, 160
DURACAO = FIM - INICIO + 1
IT_FUNDO = 3000

assert py.exists(), "Ambiente Python 3.12 ausente: rode a célula de restauração."
assert (dados / "transforms_train.json").exists() and (
    dados / "transforms_test.json"
).exists(), "Rode antes o teste mínimo (cria as câmeras)."
for sub in ["fundo", "fisico"]:
    p = modelos / sub
    if p.exists() and any(p.iterdir()):
        raise RuntimeError(f"Já existe resultado em {p}. Nada foi sobrescrito.")
modelos.mkdir(parents=True, exist_ok=True)


def cfg(oficial, nome, mudancas):
    c = json.loads((repo / "configs" / oficial).read_text())
    c.pop("test_iteration", None)
    c.update(mudancas)
    p = cfg_dir / nome
    p.write_text(json.dumps(c, indent=2))
    return p


comum = {
    "resolution": 2,
    "train_views": "0134",
    "train_views_fake": "",
    "test_views_fake": "",
    "use_refined_fake": False,
}
cfg_bg = cfg(
    "fluid_nexus_smoke_background.json",
    "smoke_20_160_fundo.json",
    {**comum, "iterations": IT_FUNDO, "save_iterations": [IT_FUNDO], "test_iterations": [IT_FUNDO]},
)
cfg_fis = cfg(
    "fluid_nexus_smoke_dynamics.json",
    "smoke_20_160_fisico.json",
    {
        **comum,
        "start_time": INICIO,
        "duration": DURACAO,
        "time_step": 1,
        "bg_load_iteration": IT_FUNDO,
        "iterations_per_time_first": 200,
        "iterations_per_time_current": 50,
        "iterations_per_time_current_max": 50,
        "iterations_per_time_current_sparse": 50,
        "iterations_per_time_current_level_two": 50,
        "iterations_per_time_current_level_two_max": 50,
        "save_iterations": [200, 500, 1000, 2200],
        "test_iterations": [200, 500, 1000, 2200],
        "spherical_cam_start": INICIO,
        "spherical_cam_duration": DURACAO,
    },
)

ambiente = os.environ.copy()
ambiente.update({"CUDA_HOME": "/usr/local/cuda-13.0", "TORCH_CUDA_ARCH_LIST": "7.5", "PYTHONUNBUFFERED": "1"})
barra = re.compile(r"\d+%\|")


def rodar(nome, args):
    log_local = Path(f"/content/{nome}.txt")
    t0 = time.time()
    print(
        "\n" + "=" * 70 + f'\n{nome}  (início {datetime.now().strftime("%H:%M:%S")} UTC)\n' + "=" * 70,
        flush=True,
    )
    ultimo = None
    with log_local.open("w") as arq:
        p = subprocess.Popen(
            [str(py), "-u"] + [str(a) for a in args],
            cwd=repo,
            env=ambiente,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for linha in p.stdout:
            arq.write(linha)
            arq.flush()
            if barra.search(linha):
                m = re.search(r"(\d+)/(\d+) \[", linha)
                if m and linha.lstrip().startswith(("Simulating", "Training", "Optimizing current")):
                    k, n = int(m.group(1)), int(m.group(2))
                    if k != ultimo and (k % 10 == 0 or k == n):
                        ultimo = k
                        print(f"  progresso: {k}/{n}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
                continue
            if linha.strip():
                print(linha.rstrip()[:200], flush=True)
        codigo = p.wait()
    minutos = (time.time() - t0) / 60
    try:
        shutil.copy2(log_local, projeto / "logs" / log_local.name)
    except Exception as e:
        print("Aviso: log não copiado para o Drive:", e)
    print(f">>> {nome}: código {codigo} em {minutos:.1f} min", flush=True)
    return codigo, minutos


c1, t1 = rodar(
    "smoke_20_160_fundo",
    [
        "entries_fluid_nexus/train_background.py",
        "--data_path",
        dados,
        "--config_path",
        cfg_bg,
        "--loader",
        "fluid_nexus_real",
        "--model_path",
        modelos / "fundo",
    ],
)
if c1 != 0:
    raise RuntimeError("O treino do fundo falhou. Veja logs/smoke_20_160_fundo.txt")
c2, t2 = rodar(
    "smoke_20_160_fisico",
    [
        "entries_fluid_nexus/train_physical_particle.py",
        "--loader",
        "fluid_nexus_real",
        "--data_path",
        dados,
        "--config_path",
        cfg_fis,
        "--bg_load_path",
        modelos / "fundo",
        "--model_path",
        modelos / "fisico",
    ],
)
(modelos / "tempos.json").write_text(
    json.dumps({"fundo_min": round(t1, 1), "fisico_min": round(t2, 1), "gpu": "T4"}, indent=2)
)
if c2 != 0:
    raise RuntimeError("O treino físico falhou. Veja logs/smoke_20_160_fisico.txt")
print(f"\nFUNDO + FÍSICO DO SMOKE CONCLUÍDOS: fundo {t1:.1f} min, físico {t2:.1f} min")


In [ ]:
#@title [Smoke] Treinar a reconstrução física em meia resolução e passo 2 (frames 20 a 160, configuração leve)
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import json, os, re, shutil, subprocess, time

projeto = Path("/content/drive/MyDrive/TCC2_FluidNexus")
repo = projeto / "codigo" / "FluidNexus" / "FluidDynamics"
py = Path("/content/fluidnexus_py312/bin/python")
origem = projeto / "dados" / "extraidos" / "FluidNexusSmoke"
dados = projeto / "dados" / "processados" / "Smoke_meia_resolucao"
cfg_dir = projeto / "configuracoes"
modelos = projeto / "modelos" / "Smoke_20_160"
INICIO, FIM, PASSO = 20, 160, 2
DURACAO = (FIM - INICIO) // PASSO + 1
IT_FUNDO = 3000

if not os.path.ismount("/content/drive"):
    from google.colab import drive

    drive.mount("/content/drive")
assert py.exists(), "Ambiente Python 3.12 ausente: rode a célula de restauração."
assert (
    modelos / "fundo" / "point_cloud" / f"iteration_{IT_FUNDO:05d}"
).exists(), "Fundo treinado não encontrado."

fis = modelos / "fisico"
if fis.exists() and any(fis.iterdir()):
    destino = modelos / f'fisico_falhou_ram_{datetime.now().strftime("%H%M%S")}'
    fis.rename(destino)
    print("Tentativa anterior guardada em:", destino)

t0 = time.time()
tarefas = []
for cam in range(5):
    (dados / f"camera{cam:02d}").mkdir(parents=True, exist_ok=True)
    for f in range(0, FIM + 1):
        tarefas.append(
            (origem / f"camera{cam:02d}" / f"{f:03d}.png", dados / f"camera{cam:02d}" / f"{f:03d}.png")
        )


def reduzir(par):
    src, dst = par
    if dst.exists() and dst.stat().st_size > 0:
        return 0
    im = Image.open(src).convert("RGB")
    im.resize((im.width // 2, im.height // 2), Image.Resampling.LANCZOS).save(dst)
    return 1


with ThreadPoolExecutor(8) as ex:
    novas = sum(ex.map(reduzir, tarefas))
for nome in ["transforms.json", "transforms_train.json", "transforms_test.json"]:
    shutil.copy2(origem / nome, dados / nome)
exemplo = Image.open(dados / "camera02" / "040.png")
print(
    f"Imagens reduzidas: {len(tarefas)} (novas: {novas}) em {(time.time() - t0) / 60:.1f} min | tamanho: {exemplo.size}"
)

c = json.loads((repo / "configs" / "fluid_nexus_smoke_dynamics.json").read_text())
c.pop("test_iteration", None)
c.update(
    {
        "resolution": 1,
        "train_views": "0134",
        "train_views_fake": "",
        "test_views_fake": "",
        "use_refined_fake": False,
        "start_time": INICIO,
        "duration": DURACAO,
        "time_step": PASSO,
        "bg_load_iteration": IT_FUNDO,
        "iterations_per_time_first": 200,
        "iterations_per_time_current": 50,
        "iterations_per_time_current_max": 50,
        "iterations_per_time_current_sparse": 50,
        "iterations_per_time_current_level_two": 50,
        "iterations_per_time_current_level_two_max": 50,
        "save_iterations": [200, 500, 1000, 2200],
        "test_iterations": [200, 500, 1000, 2200],
        "spherical_cam_start": INICIO,
        "spherical_cam_duration": DURACAO,
    }
)
cfg_fis = cfg_dir / "smoke_20_160_fisico_meia_res_passo2.json"
cfg_fis.write_text(json.dumps(c, indent=2))

ambiente = os.environ.copy()
ambiente.update({"CUDA_HOME": "/usr/local/cuda-13.0", "TORCH_CUDA_ARCH_LIST": "7.5", "PYTHONUNBUFFERED": "1"})
barra = re.compile(r"\d+%\|")
log_local = Path("/content/smoke_20_160_fisico.txt")
t0 = time.time()
ultimo = None
print(
    "\n"
    + "=" * 70
    + f'\nFÍSICO SMOKE 20-160 (início {datetime.now().strftime("%H:%M:%S")} UTC)\n'
    + "=" * 70,
    flush=True,
)
with log_local.open("w") as arq:
    p = subprocess.Popen(
        [
            str(py),
            "-u",
            "entries_fluid_nexus/train_physical_particle.py",
            "--loader",
            "fluid_nexus_real",
            "--data_path",
            str(dados),
            "--config_path",
            str(cfg_fis),
            "--bg_load_path",
            str(modelos / "fundo"),
            "--model_path",
            str(fis),
        ],
        cwd=repo,
        env=ambiente,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for linha in p.stdout:
        arq.write(linha)
        arq.flush()
        if barra.search(linha):
            m = re.search(r"(\d+)/(\d+) \[", linha)
            if m and linha.lstrip().startswith("Simulating"):
                k, n = int(m.group(1)), int(m.group(2))
                if k != ultimo and (k % 10 == 0 or k == n):
                    ultimo = k
                    print(f"  progresso: {k}/{n}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
            continue
        if linha.strip() and not linha.startswith(("visual_", "hidden_", "center_")):
            print(linha.rstrip()[:200], flush=True)
    codigo = p.wait()
minutos = (time.time() - t0) / 60
try:
    shutil.copy2(log_local, projeto / "logs" / log_local.name)
except Exception as e:
    print("Aviso: log não copiado:", e)
(modelos / "tempos.json").write_text(
    json.dumps({"fundo_min": 1.3, "fisico_min": round(minutos, 1), "gpu": "T4"}, indent=2)
)
if codigo != 0:
    raise RuntimeError(f"O treino físico falhou (código {codigo}). Veja logs/smoke_20_160_fisico.txt")
print(f"\nFÍSICO DO SMOKE CONCLUÍDO em {minutos:.1f} min")


In [ ]:
#@title [Smoke] Treinar a reconstrução visual (frames 20 a 160, passo 2, configuração leve)
from pathlib import Path
from datetime import datetime
import json, os, re, shutil, subprocess, time

projeto = Path("/content/drive/MyDrive/TCC2_FluidNexus")
repo = projeto / "codigo" / "FluidNexus" / "FluidDynamics"
py = Path("/content/fluidnexus_py312/bin/python")
dados = projeto / "dados" / "processados" / "Smoke_meia_resolucao"
modelos = projeto / "modelos" / "Smoke_20_160"
cfg_fis = projeto / "configuracoes" / "smoke_20_160_fisico_meia_res_passo2.json"
cfg_vis = projeto / "configuracoes" / "smoke_20_160_visual_meia_res_passo2.json"
fis, vis = modelos / "fisico", modelos / "visual"

if not os.path.ismount("/content/drive"):
    from google.colab import drive

    drive.mount("/content/drive")
assert py.exists(), "Ambiente Python 3.12 ausente: rode a célula de restauração."
assert (fis / "checkpoint").exists(), f"Checkpoints físicos não encontrados em {fis}"
if vis.exists() and any(vis.iterdir()):
    vis.rename(modelos / f'visual_anterior_{datetime.now().strftime("%H%M%S")}')
vis.mkdir(parents=True, exist_ok=True)
shutil.copy2(cfg_fis, cfg_vis)

ambiente = os.environ.copy()
ambiente.update({"CUDA_HOME": "/usr/local/cuda-13.0", "TORCH_CUDA_ARCH_LIST": "7.5", "PYTHONUNBUFFERED": "1"})
barra = re.compile(r"\d+%\|")
log_local = Path("/content/smoke_20_160_visual.txt")
t0 = time.time()
ultimo = None
print(
    "=" * 70 + f'\nVISUAL SMOKE 20-160 (início {datetime.now().strftime("%H:%M:%S")} UTC)\n' + "=" * 70,
    flush=True,
)
with log_local.open("w") as arq:
    p = subprocess.Popen(
        [
            str(py),
            "-u",
            "entries_fluid_nexus/train_visual_particle.py",
            "--loader",
            "fluid_nexus_real",
            "--data_path",
            str(dados),
            "--config_path",
            str(cfg_vis),
            "--bg_load_path",
            str(modelos / "fundo"),
            "--load_path",
            str(fis),
            "--model_path",
            str(vis),
        ],
        cwd=repo,
        env=ambiente,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for linha in p.stdout:
        arq.write(linha)
        arq.flush()
        if barra.search(linha):
            m = re.search(r"(\d+)/(\d+) \[", linha)
            if m and linha.lstrip().startswith(("Simulating", "Optimizing current", "Training")):
                k, n = int(m.group(1)), int(m.group(2))
                if k != ultimo and (k % 10 == 0 or k == n):
                    ultimo = k
                    print(f"  progresso: {k}/{n}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
            continue
        if linha.strip() and not linha.startswith(("visual_", "hidden_", "center_", "       ")):
            print(linha.rstrip()[:200], flush=True)
    codigo = p.wait()
minutos = (time.time() - t0) / 60
try:
    shutil.copy2(log_local, projeto / "logs" / log_local.name)
except Exception as e:
    print("Aviso: log não copiado:", e)
tempos = json.loads((modelos / "tempos.json").read_text()) if (modelos / "tempos.json").exists() else {}
tempos["visual_min"] = round(minutos, 1)
(modelos / "tempos.json").write_text(json.dumps(tempos, indent=2))
if codigo != 0:
    raise RuntimeError(f"O treino visual falhou (código {codigo}). Veja logs/smoke_20_160_visual.txt")
print(f"\nVISUAL DO SMOKE CONCLUÍDO em {minutos:.1f} min")


In [ ]:
#@title [Ball] Pipeline completo na configuração leve: câmeras + meia resolução + fundo + físico + visual (frames 20 a 160, passo 2)
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import json, os, re, shutil, subprocess, time

projeto = Path("/content/drive/MyDrive/TCC2_FluidNexus")
repo = projeto / "codigo" / "FluidNexus" / "FluidDynamics"
py = Path("/content/fluidnexus_py312/bin/python")
origem = projeto / "dados" / "extraidos" / "FluidNexusBall"
dados = projeto / "dados" / "processados" / "Ball_meia_resolucao"
cfg_dir = projeto / "configuracoes"
modelos = projeto / "modelos" / "Ball_20_160"
fundo, fis, vis = modelos / "fundo", modelos / "fisico", modelos / "visual"
INICIO, FIM, PASSO = 20, 160, 2
DURACAO = (FIM - INICIO) // PASSO + 1
IT_FUNDO = 3000

if not os.path.ismount("/content/drive"):
    from google.colab import drive

    drive.mount("/content/drive")
assert py.exists(), "Ambiente Python 3.12 ausente: rode a célula de restauração."
modelos.mkdir(parents=True, exist_ok=True)

base = json.loads((origem / "transforms.json").read_text())
for nome, cams in [("transforms_train.json", "0134"), ("transforms_test.json", "2")]:
    c = {k: v for k, v in base.items() if k != "frames"}
    c["frames"] = [f for f in base["frames"] if f["file_path"][-1] in cams]
    (origem / nome).write_text(json.dumps(c, indent=2))
print(
    "Câmeras de treino:",
    [f["file_path"] for f in base["frames"] if f["file_path"][-1] in "0134"],
    "| teste: camera02",
)

t0 = time.time()
tarefas = []
for cam in range(5):
    (dados / f"camera{cam:02d}").mkdir(parents=True, exist_ok=True)
    tarefas += [
        (origem / f"camera{cam:02d}" / f"{f:03d}.png", dados / f"camera{cam:02d}" / f"{f:03d}.png")
        for f in range(FIM + 1)
    ]


def reduzir(par):
    src, dst = par
    if dst.exists() and dst.stat().st_size > 0:
        return 0
    im = Image.open(src).convert("RGB")
    im.resize((im.width // 2, im.height // 2), Image.Resampling.LANCZOS).save(dst)
    return 1


with ThreadPoolExecutor(8) as ex:
    novas = sum(ex.map(reduzir, tarefas))
for nome in ["transforms.json", "transforms_train.json", "transforms_test.json"]:
    shutil.copy2(origem / nome, dados / nome)
print(f"Imagens em meia resolução: {len(tarefas)} (novas: {novas}) em {(time.time() - t0) / 60:.1f} min")


def oficial(*nomes):
    for n in nomes:
        if (repo / "configs" / n).exists():
            return n
    raise FileNotFoundError(nomes)


comum = {"train_views": "0134", "train_views_fake": "", "test_views_fake": "", "use_refined_fake": False}
c = json.loads((repo / "configs" / "fluid_nexus_ball_background.json").read_text())
c.pop("test_iteration", None)
c.update(
    {
        **comum,
        "resolution": 2,
        "iterations": IT_FUNDO,
        "save_iterations": [IT_FUNDO],
        "test_iterations": [IT_FUNDO],
    }
)
cfg_bg = cfg_dir / "ball_20_160_fundo.json"
cfg_bg.write_text(json.dumps(c, indent=2))
leve = {
    **comum,
    "resolution": 1,
    "start_time": INICIO,
    "duration": DURACAO,
    "time_step": PASSO,
    "bg_load_iteration": IT_FUNDO,
    "iterations_per_time_first": 200,
    "iterations_per_time_current": 50,
    "iterations_per_time_current_max": 50,
    "iterations_per_time_current_sparse": 50,
    "iterations_per_time_current_level_two": 50,
    "iterations_per_time_current_level_two_max": 50,
    "save_iterations": [200, 500, 1000, 2200],
    "test_iterations": [200, 500, 1000, 2200],
    "spherical_cam_start": INICIO,
    "spherical_cam_duration": DURACAO,
}
cfgs = {}
for etapa, candidatos in [
    ("fisico", ["fluid_nexus_ball.json", "fluid_nexus_ball_dynamics.json"]),
    ("visual", ["fluid_nexus_ball_visual.json", "fluid_nexus_ball.json", "fluid_nexus_ball_dynamics.json"]),
]:
    n = oficial(*candidatos)
    c = json.loads((repo / "configs" / n).read_text())
    c.pop("test_iteration", None)
    c.update(leve)
    cfgs[etapa] = cfg_dir / f"ball_20_160_{etapa}.json"
    cfgs[etapa].write_text(json.dumps(c, indent=2))
    print(f"Config {etapa}: baseada em {n}")

ambiente = os.environ.copy()
ambiente.update({"CUDA_HOME": "/usr/local/cuda-13.0", "TORCH_CUDA_ARCH_LIST": "7.5", "PYTHONUNBUFFERED": "1"})
barra = re.compile(r"\d+%\|")
tempos = json.loads((modelos / "tempos.json").read_text()) if (modelos / "tempos.json").exists() else {}


def rodar(nome, args, saida):
    if saida.exists() and any(saida.iterdir()):
        saida.rename(saida.parent / f'{saida.name}_anterior_{datetime.now().strftime("%H%M%S")}')
    log_local = Path(f"/content/{nome}.txt")
    t0 = time.time()
    ultimo = None
    print(
        "\n" + "=" * 70 + f'\n{nome} (início {datetime.now().strftime("%H:%M:%S")} UTC)\n' + "=" * 70,
        flush=True,
    )
    with log_local.open("w") as arq:
        p = subprocess.Popen(
            [str(py), "-u"] + [str(a) for a in args],
            cwd=repo,
            env=ambiente,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for linha in p.stdout:
            arq.write(linha)
            arq.flush()
            if barra.search(linha):
                m = re.search(r"(\d+)/(\d+) \[", linha)
                if m and linha.lstrip().startswith(("Simulating", "Training", "Optimizing current")):
                    k, n = int(m.group(1)), int(m.group(2))
                    passo_print = 500 if n >= 1000 else 10
                    if k != ultimo and (k % passo_print == 0 or k == n):
                        ultimo = k
                        print(f"  progresso: {k}/{n}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
                continue
            if linha.strip() and not linha.startswith(("visual_", "hidden_", "center_", "       ")):
                print(linha.rstrip()[:200], flush=True)
        codigo = p.wait()
    minutos = (time.time() - t0) / 60
    try:
        shutil.copy2(log_local, projeto / "logs" / log_local.name)
    except Exception as e:
        print("Aviso: log não copiado:", e)
    tempos[nome] = round(minutos, 1)
    (modelos / "tempos.json").write_text(json.dumps(tempos, indent=2))
    print(f">>> {nome}: código {codigo} em {minutos:.1f} min", flush=True)
    if codigo != 0:
        raise RuntimeError(f"{nome} falhou. Veja logs/{log_local.name}")


if not (fundo / "point_cloud" / f"iteration_{IT_FUNDO:05d}").exists():
    rodar(
        "ball_20_160_fundo",
        [
            "entries_fluid_nexus/train_background.py",
            "--data_path",
            origem,
            "--config_path",
            cfg_bg,
            "--loader",
            "fluid_nexus_real",
            "--model_path",
            fundo,
        ],
        fundo,
    )
else:
    print("Fundo já pronto, pulando.")
if not (fis / "checkpoint").exists():
    rodar(
        "ball_20_160_fisico",
        [
            "entries_fluid_nexus/train_physical_particle.py",
            "--loader",
            "fluid_nexus_real",
            "--data_path",
            dados,
            "--config_path",
            cfgs["fisico"],
            "--bg_load_path",
            fundo,
            "--model_path",
            fis,
        ],
        fis,
    )
else:
    print("Físico já pronto, pulando.")
rodar(
    "ball_20_160_visual",
    [
        "entries_fluid_nexus/train_visual_particle.py",
        "--loader",
        "fluid_nexus_real",
        "--data_path",
        dados,
        "--config_path",
        cfgs["visual"],
        "--bg_load_path",
        fundo,
        "--load_path",
        fis,
        "--model_path",
        vis,
    ],
    vis,
)
print("\nBALL COMPLETO (fundo + físico + visual):", tempos)


## 4. Reconstrução 3D: versão alta

In [ ]:
#@title [Versão alta] Comparar configurações oficiais x versão leve (só leitura)
import json
from pathlib import Path

REPO = Path("/content/drive/MyDrive/TCC2_FluidNexus/codigo/FluidNexus/FluidDynamics")
CFG_LEVE = Path("/content/drive/MyDrive/TCC2_FluidNexus/configuracoes")
CHAVES = [
    "resolution",
    "start_time",
    "duration",
    "time_step",
    "iterations",
    "iterations_per_time_first",
    "iterations_per_time_current",
    "iterations_per_time_current_max",
    "iterations_per_time_current_sparse",
    "iterations_per_time_current_level_two",
    "bg_load_iteration",
    "train_views",
    "model",
    "rd_pipe",
]


def resumo(p):
    try:
        d = json.loads(p.read_text())
    except Exception as e:
        return f"erro: {e}"
    return {k: d[k] for k in CHAVES if k in d}


print("=== CONFIGURAÇÕES OFICIAIS DO REPOSITÓRIO ===")
oficiais = sorted(
    p
    for p in REPO.rglob("*.json")
    if any(t in str(p).lower() for t in ["scalar", "smoke", "ball"])
    and "transforms" not in p.name
    and "submodules" not in str(p)
)
for p in oficiais:
    print("\n" + str(p.relative_to(REPO)))
    print(resumo(p))

print("\n\n=== CONFIGURAÇÕES DA VERSÃO LEVE (nossas) ===")
for p in sorted(CFG_LEVE.glob("*20_160*.json")):
    print("\n" + p.name)
    print(resumo(p))


In [ ]:
#@title [Versão alta] ScalarReal: treino físico + visual (configuração oficial, resolução cheia, frames 20 a 160)
from pathlib import Path
from datetime import datetime, timezone
import json, os, re, shutil, subprocess, time

projeto = Path("/content/drive/MyDrive/TCC2_FluidNexus")
repo = projeto / "codigo" / "FluidNexus" / "FluidDynamics"
python_env = Path("/content/fluidnexus_py312/bin/python")
dados = projeto / "dados" / "processados" / "ScalarReal_20_160"
base = projeto / "modelos" / "ScalarReal_20_160_alta"
modelo_fisico, modelo_visual = base / "fisico", base / "visual"
cfg_leve = projeto / "configuracoes" / "scalar_real_20_160_fisico.json"
cfg_oficial = repo / "configs" / "scalar_real.json"
cfg_fis = projeto / "configuracoes" / "scalar_real_20_160_alta_fisico.json"
cfg_vis = projeto / "configuracoes" / "scalar_real_20_160_alta_visual.json"

assert python_env.exists(), "Rode antes a célula de restauração do ambiente."
for pasta in [modelo_fisico, modelo_visual, projeto / "logs"]:
    pasta.mkdir(parents=True, exist_ok=True)

cap = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
).split()[0]
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()

oficial = json.loads(cfg_oficial.read_text())
cfg = json.loads(cfg_leve.read_text())
cfg.update(
    {
        "resolution": oficial["resolution"],
        "iterations_per_time_first": oficial["iterations_per_time_first"],
        "save_iterations": oficial.get("save_iterations", cfg["save_iterations"]),
        "test_iterations": oficial.get("test_iterations", cfg["test_iterations"]),
        "model_path": str(modelo_fisico),
    }
)
for k in [
    "iterations_per_time_current",
    "iterations_per_time_current_max",
    "iterations_per_time_current_sparse",
    "iterations_per_time_current_level_two",
    "iterations_per_time_current_level_two_max",
]:
    if k in oficial:
        cfg[k] = oficial[k]
cfg_fis.write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
cfg_v = dict(cfg, model_path=str(modelo_visual), load_path=str(modelo_fisico))
cfg_vis.write_text(json.dumps(cfg_v, indent=2, ensure_ascii=False))
print(
    "Configuração alta:",
    {
        k: cfg[k]
        for k in [
            "resolution",
            "start_time",
            "duration",
            "time_step",
            "iterations_per_time_first",
            "iterations_per_time_current",
            "train_views",
        ]
    },
)

ambiente = os.environ.copy()
ambiente.update({"CUDA_HOME": "/usr/local/cuda-13.0", "TORCH_CUDA_ARCH_LIST": cap, "PYTHONUNBUFFERED": "1"})
barra = re.compile(r"\d+%\|")
tempos_path = base / "tempos.json"
tempos = json.loads(tempos_path.read_text()) if tempos_path.exists() else {}
tempos["gpu"] = gpu


def etapa(nome, script, saida, cfg_path, extra=()):
    if (saida / "checkpoint").exists() and nome in tempos:
        print(f"\n{nome}: já concluído antes ({tempos[nome]} min). Pulando.")
        return
    if saida.exists() and any(saida.iterdir()):
        shutil.rmtree(saida)
        print(f"Pasta incompleta de uma execução interrompida foi limpa: {saida}")
    saida.mkdir(parents=True, exist_ok=True)
    log_local = Path(f"/content/log_scalarreal_alta_{nome}.txt")
    log_drive = projeto / "logs" / f"treinamento_{nome}_scalarreal_20_160_alta.txt"
    comando = [
        str(python_env),
        "-u",
        str(repo / "entries_scalar_real" / script),
        "--data_path",
        str(dados),
        "--model_path",
        str(saida),
        "--loader",
        "scalar_real",
        "--config_path",
        str(cfg_path),
        *extra,
    ]
    print(
        "\n" + "=" * 78 + f"\nSCALARREAL VERSÃO ALTA: {nome.upper()} | GPU {gpu}\n"
        f"Início UTC: {datetime.now(timezone.utc).isoformat()}\n" + "=" * 78,
        flush=True,
    )
    t0, ultimo, ultima_copia = time.time(), None, time.time()
    with log_local.open("w", encoding="utf-8") as arq:
        proc = subprocess.Popen(
            comando,
            cwd=repo,
            env=ambiente,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for linha in proc.stdout:
            arq.write(linha)
            if barra.search(linha):
                m = re.search(r"(\d+)/(\d+) \[", linha)
                if linha.lstrip().startswith(("Simulating", "Optimizing current", "Training")) and m:
                    passo = int(m.group(1))
                    if passo != ultimo and (passo % 10 == 0 or passo == int(m.group(2))):
                        ultimo = passo
                        print(
                            f"  progresso: {passo}/{m.group(2)} | {(time.time()-t0)/60:.1f} min", flush=True
                        )
            elif linha.strip():
                print(linha.rstrip()[:200], flush=True)
            if time.time() - ultima_copia > 300:
                arq.flush()
                shutil.copy(log_local, log_drive)
                ultima_copia = time.time()
        codigo = proc.wait()
    shutil.copy(log_local, log_drive)
    minutos = round((time.time() - t0) / 60, 1)
    print(f"{nome}: código {codigo} | {minutos} min", flush=True)
    if codigo != 0:
        raise RuntimeError(f"{nome} falhou. Log: {log_drive}")
    tempos[nome] = minutos
    tempos_path.write_text(json.dumps(tempos, indent=2))


etapa("fisico", "train_physical_particle.py", modelo_fisico, cfg_fis)
etapa("visual", "train_visual_particle.py", modelo_visual, cfg_vis, ("--load_path", str(modelo_fisico)))
print("\nSCALARREAL VERSÃO ALTA CONCLUÍDO:", tempos)


In [ ]:
#@title [Versão alta] Smoke + Ball: fundo + físico + visual (configuração oficial, resolução cheia, frames 20 a 160)
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor
import json, os, re, shutil, subprocess, time

projeto = Path("/content/drive/MyDrive/TCC2_FluidNexus")
repo = projeto / "codigo" / "FluidNexus" / "FluidDynamics"
python_env = Path("/content/fluidnexus_py312/bin/python")
assert python_env.exists(), "Rode antes a célula de restauração do ambiente."
cap = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
).split()[0]
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
ambiente = os.environ.copy()
ambiente.update({"CUDA_HOME": "/usr/local/cuda-13.0", "TORCH_CUDA_ARCH_LIST": cap, "PYTHONUNBUFFERED": "1"})
barra = re.compile(r"\d+%\|")
FIM = 160

FLUIDOS = {
    "Smoke": dict(
        origem="FluidNexusSmoke",
        oficial="fluid_nexus_smoke",
        leve_fundo="smoke_20_160_fundo.json",
        leve_din="smoke_20_160_fisico_meia_res_passo2.json",
    ),
    "Ball": dict(
        origem="FluidNexusBall",
        oficial="fluid_nexus_ball",
        leve_fundo="ball_20_160_fundo.json",
        leve_din="ball_20_160_fisico.json",
    ),
}
CHAVES_IT = [
    "iterations_per_time_first",
    "iterations_per_time_current",
    "iterations_per_time_current_max",
    "iterations_per_time_current_sparse",
    "iterations_per_time_current_level_two",
    "iterations_per_time_current_level_two_max",
]


def copiar_local(origem):
    destino = Path("/content/dados_alta") / origem.name
    tarefas = []
    for cam in sorted(origem.glob("camera0*")):
        for img in cam.glob("*.png"):
            if img.stem.isdigit() and int(img.stem) <= FIM:
                alvo = destino / cam.name / img.name
                if not alvo.exists():
                    tarefas.append((img, alvo))
    for js in origem.glob("*.json"):
        destino.mkdir(parents=True, exist_ok=True)
        shutil.copy2(js, destino / js.name)
    t0 = time.time()
    for _, alvo in tarefas:
        alvo.parent.mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(16) as ex:
        list(ex.map(lambda t: shutil.copy2(*t), tarefas))
    total = len(list(destino.glob("camera0*/*.png")))
    print(
        f"Imagens locais em {destino}: {total} (copiadas agora: {len(tarefas)} em {(time.time()-t0)/60:.1f} min)"
    )
    return destino


def rodar(rotulo, nome, comando, saida, tempos, tempos_path):
    if nome in tempos:
        print(f"\n{rotulo} {nome}: já concluído antes ({tempos[nome]} min). Pulando.")
        return
    if saida.exists() and any(saida.iterdir()):
        shutil.rmtree(saida)
        print(f"Pasta incompleta de uma execução interrompida foi limpa: {saida}")
    saida.mkdir(parents=True, exist_ok=True)
    log_local = Path(f"/content/log_{rotulo.lower()}_alta_{nome}.txt")
    log_drive = projeto / "logs" / f"treinamento_{nome}_{rotulo.lower()}_20_160_alta.txt"
    print(
        "\n" + "=" * 78 + f"\n{rotulo.upper()} VERSÃO ALTA: {nome.upper()} | GPU {gpu}\n"
        f"Início UTC: {datetime.now(timezone.utc).isoformat()}\n" + "=" * 78,
        flush=True,
    )
    t0, ultimo, ultima_copia = time.time(), None, time.time()
    with log_local.open("w", encoding="utf-8") as arq:
        proc = subprocess.Popen(
            [str(c) for c in comando],
            cwd=repo,
            env=ambiente,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for linha in proc.stdout:
            arq.write(linha)
            if barra.search(linha):
                m = re.search(r"(\d+)/(\d+) \[", linha)
                if m and linha.lstrip().startswith(("Simulating", "Optimizing current", "Training")):
                    passo, total = int(m.group(1)), int(m.group(2))
                    marca = max(1, total // 14)
                    if passo != ultimo and (passo % marca == 0 or passo == total):
                        ultimo = passo
                        print(f"  progresso: {passo}/{total} | {(time.time()-t0)/60:.1f} min", flush=True)
            elif linha.strip():
                print(linha.rstrip()[:200], flush=True)
            if time.time() - ultima_copia > 300:
                arq.flush()
                shutil.copy(log_local, log_drive)
                ultima_copia = time.time()
        codigo = proc.wait()
    shutil.copy(log_local, log_drive)
    minutos = round((time.time() - t0) / 60, 1)
    print(f"{rotulo} {nome}: código {codigo} | {minutos} min", flush=True)
    if codigo != 0:
        raise RuntimeError(f"{rotulo} {nome} falhou. Log: {log_drive}")
    tempos[nome] = minutos
    tempos_path.write_text(json.dumps(tempos, indent=2))


for rotulo, f in FLUIDOS.items():
    print("\n" + "#" * 78 + f"\n# {rotulo}\n" + "#" * 78, flush=True)
    dados = copiar_local(projeto / "dados" / "extraidos" / f["origem"])
    base = projeto / "modelos" / f"{rotulo}_20_160_alta"
    fundo, fisico, visual = base / "fundo", base / "fisico", base / "visual"
    cfgs = projeto / "configuracoes"
    of_bg = json.loads((repo / "configs" / f"{f['oficial']}_background.json").read_text())
    of_din = json.loads((repo / "configs" / f"{f['oficial']}_dynamics.json").read_text())

    bg = json.loads((cfgs / f["leve_fundo"]).read_text())
    bg.update(
        {
            "resolution": of_bg["resolution"],
            "iterations": of_bg["iterations"],
            "save_iterations": [of_bg["iterations"]],
            "test_iterations": [of_bg["iterations"]],
        }
    )
    din = json.loads((cfgs / f["leve_din"]).read_text())
    din.update(
        {
            "resolution": of_din["resolution"],
            "bg_load_iteration": of_bg["iterations"],
            "save_iterations": of_din.get("save_iterations", din["save_iterations"]),
            "test_iterations": of_din.get("test_iterations", din["test_iterations"]),
        }
    )
    for k in CHAVES_IT:
        if k in of_din:
            din[k] = of_din[k]
    for d in (bg, din):
        for k in ["model_path", "data_path", "load_path", "bg_load_path", "loader"]:
            d.pop(k, None)
    cfg_bg = cfgs / f"{rotulo.lower()}_20_160_alta_fundo.json"
    cfg_din = cfgs / f"{rotulo.lower()}_20_160_alta_dinamica.json"
    cfg_bg.write_text(json.dumps(bg, indent=2, ensure_ascii=False))
    cfg_din.write_text(json.dumps(din, indent=2, ensure_ascii=False))
    print("Fundo:", {k: bg[k] for k in ["resolution", "iterations", "train_views"]})
    print(
        "Dinâmica:",
        {
            k: din[k]
            for k in [
                "resolution",
                "start_time",
                "duration",
                "time_step",
                "iterations_per_time_first",
                "iterations_per_time_current",
                "bg_load_iteration",
                "train_views",
            ]
        },
    )

    base.mkdir(parents=True, exist_ok=True)
    tempos_path = base / "tempos.json"
    tempos = json.loads(tempos_path.read_text()) if tempos_path.exists() else {}
    tempos["gpu"] = gpu
    ent = repo / "entries_fluid_nexus"
    comum = ["--loader", "fluid_nexus_real", "--data_path", dados]
    rodar(
        rotulo,
        "fundo",
        [
            python_env,
            "-u",
            ent / "train_background.py",
            *comum,
            "--config_path",
            cfg_bg,
            "--model_path",
            fundo,
        ],
        fundo,
        tempos,
        tempos_path,
    )
    rodar(
        rotulo,
        "fisico",
        [
            python_env,
            "-u",
            ent / "train_physical_particle.py",
            *comum,
            "--config_path",
            cfg_din,
            "--bg_load_path",
            fundo,
            "--model_path",
            fisico,
        ],
        fisico,
        tempos,
        tempos_path,
    )
    rodar(
        rotulo,
        "visual",
        [
            python_env,
            "-u",
            ent / "train_visual_particle.py",
            *comum,
            "--config_path",
            cfg_din,
            "--bg_load_path",
            fundo,
            "--load_path",
            fisico,
            "--model_path",
            visual,
        ],
        visual,
        tempos,
        tempos_path,
    )
    print(f"\n{rotulo.upper()} VERSÃO ALTA CONCLUÍDO:", tempos, flush=True)

print("\nSMOKE E BALL VERSÃO ALTA CONCLUÍDOS")


## 5. Avaliação da reconstrução e vídeos

In [ ]:
#@title [Leve x Alta] Comparação da qualidade: Tabela 1 por região + figuras dos frames 40, 80, 120 e 160
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import os, subprocess, sys, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage
from skimage.metrics import structural_similarity
from skimage.morphology import remove_small_objects
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
try:
    import lpips, torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)
    import lpips, torch

raiz = Path("/content/drive/MyDrive/TCC2_FluidNexus")
saida = raiz / "resultados" / "comparacao_leve_x_alta"
saida.mkdir(parents=True, exist_ok=True)
pasta_fig = raiz / "figuras"
FRAMES_DESTAQUE = [40, 80, 120, 160]
FLUIDOS = {
    "ScalarReal": dict(vista="train02", passo=1, fundo=None),
    "Smoke": dict(vista="camera02", passo=2, fundo=raiz / "dados/extraidos/FluidNexusSmoke/camera02/000.png"),
    "Ball": dict(vista="camera02", passo=2, fundo=raiz / "dados/extraidos/FluidNexusBall/camera02/000.png"),
}
INICIO, FIM = 20, 160
VERSOES = {"leve": "{f}_20_160", "alta": "{f}_20_160_alta"}
disp = "cuda" if torch.cuda.is_available() else "cpu"
modelo_lpips = lpips.LPIPS(net="alex", spatial=True, verbose=False).to(disp)


def rgb(p):
    return np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 255.0


def ajustar(ref, img):
    if ref.shape == img.shape:
        return img
    h, w = ref.shape[:2]
    return (
        np.asarray(
            Image.fromarray(np.uint8(np.clip(img, 0, 1) * 255)).resize((w, h), Image.Resampling.LANCZOS),
            dtype=np.float32,
        )
        / 255.0
    )


def psnr(a, b, m):
    mse = float(((a - b) ** 2)[m].mean())
    return 10 * np.log10(1.0 / max(mse, 1e-10))


def mapa_lpips(a, b):
    ta = torch.from_numpy(a).permute(2, 0, 1)[None].to(disp) * 2 - 1
    tb = torch.from_numpy(b).permute(2, 0, 1)[None].to(disp) * 2 - 1
    with torch.no_grad():
        return modelo_lpips(ta, tb)[0, 0].cpu().numpy()


def indice(pasta):
    return {p.name: p for p in pasta.glob("*.png")}


def achar(arqs, prefixo, i, vista):
    c = sorted(n for n in arqs if n.startswith(f"{prefixo}_frame{i:03d}_{vista}"))
    return arqs[c[-1]] if c else None


linhas, exemplos = [], {}
for nome, cfg in FLUIDOS.items():
    t0 = time.time()
    arqs = {
        v: indice(raiz / "modelos" / (molde.format(f=nome)) / "visual" / "training_render")
        for v, molde in VERSOES.items()
    }
    print(f'{nome}: renders leve={len(arqs["leve"])} | alta={len(arqs["alta"])}', flush=True)
    fundo_ref = None
    for f in range(INICIO, FIM + 1, cfg["passo"]):
        i = (f - INICIO) // cfg["passo"]
        p_real = achar(arqs["alta"], "real", i, cfg["vista"])
        caminhos = {v: achar(arqs[v], "render", i, cfg["vista"]) for v in VERSOES}
        if p_real is None or any(p is None for p in caminhos.values()):
            continue
        with ThreadPoolExecutor(3) as ex:
            real, *recs = ex.map(rgb, [p_real, *caminhos.values()])
        recs = dict(zip(caminhos, [ajustar(real, r) for r in recs]))
        if fundo_ref is None:
            fundo_ref = np.zeros_like(real) if cfg["fundo"] is None else ajustar(real, rgb(cfg["fundo"]))
        m = np.abs(real - fundo_ref).mean(axis=2) > (0.08 if cfg["fundo"] is None else 0.06)
        m = ndimage.binary_opening(m, iterations=2)
        m = remove_small_objects(m, min_size=200)
        m = ndimage.binary_dilation(m, iterations=4)
        if m.sum() < 100:
            continue
        regioes = [("imagem", np.ones_like(m)), ("fumaca", m), ("fundo", ~m)]
        for v, rec in recs.items():
            _, mapa_ssim = structural_similarity(real, rec, channel_axis=2, data_range=1.0, full=True)
            mapa_ssim = mapa_ssim.mean(axis=2)
            ml = mapa_lpips(real, rec)
            if ml.shape != m.shape:
                ml = np.asarray(
                    Image.fromarray(ml).resize((m.shape[1], m.shape[0]), Image.Resampling.BILINEAR)
                )
            linha = {"fluido": nome, "versao": v, "frame": f, "area_fumaca_%": 100 * m.mean()}
            for reg, mm in regioes:
                linha[f"psnr_{reg}"] = psnr(real, rec, mm)
                linha[f"ssim_{reg}"] = float(mapa_ssim[mm].mean())
                linha[f"lpips_{reg}"] = float(ml[mm].mean())
            linhas.append(linha)
        if f in FRAMES_DESTAQUE:
            exemplos[(nome, f)] = (real, recs["leve"], recs["alta"])
    print(
        f'{nome}: {sum(1 for l in linhas if l["fluido"] == nome) // 2} frames avaliados em {(time.time() - t0) / 60:.1f} min',
        flush=True,
    )

df = pd.DataFrame(linhas)
df.to_csv(saida / "tabela1_leve_x_alta_por_frame.csv", index=False)
cols = [c for c in df.columns if c.startswith(("psnr", "ssim", "lpips"))]
ordem = ["ScalarReal", "Smoke", "Ball"]
media = df.groupby(["fluido", "versao"])[cols].mean()
media.to_csv(saida / "tabela1_leve_x_alta_media_todos_os_frames.csv")
media_dest = df[df.frame.isin(FRAMES_DESTAQUE)].groupby(["fluido", "versao"])[cols].mean()
media_dest.to_csv(saida / "tabela1_leve_x_alta_media_frames_40_80_120_160.csv")

tab = []
for nome in ordem:
    for met in ["psnr", "ssim", "lpips"]:
        for reg, lab in [("imagem", "imagem inteira"), ("fumaca", "só fumaça"), ("fundo", "só fundo")]:
            l, a = media.loc[(nome, "leve"), f"{met}_{reg}"], media.loc[(nome, "alta"), f"{met}_{reg}"]
            tab.append(
                {
                    "fluido": nome,
                    "métrica": met.upper(),
                    "região": lab,
                    "leve": l,
                    "alta": a,
                    "ganho": (a - l) if met != "lpips" else (l - a),
                }
            )
tab = pd.DataFrame(tab)
tab.to_csv(saida / "tabela1_leve_x_alta_resumo.csv", index=False)
print('\nTABELA 1 (média de todos os frames, câmera de teste). "ganho" positivo = versão alta melhor')
display(
    tab.pivot_table(index=["fluido", "região"], columns="métrica", values=["leve", "alta", "ganho"])
    .reindex(ordem, level=0)
    .round(3)
)

fig, ax = plt.subplots(1, 3, figsize=(18, 4.8), facecolor="white")
cores = {"ScalarReal": "#1f77b4", "Smoke": "#ff7f0e", "Ball": "#2ca02c"}
for k, (met, tit) in enumerate(
    [
        ("psnr_fumaca", "PSNR só na fumaça (dB) ↑"),
        ("ssim_fumaca", "SSIM só na fumaça ↑"),
        ("lpips_fumaca", "LPIPS só na fumaça ↓"),
    ]
):
    for nome in ordem:
        for v, ls in [("leve", "--"), ("alta", "-")]:
            d = df[(df.fluido == nome) & (df.versao == v)]
            ax[k].plot(d.frame, d[met], ls=ls, color=cores[nome], lw=1.6, label=f"{nome} ({v})")
    for f in FRAMES_DESTAQUE:
        ax[k].axvline(f, color="gray", ls=":", lw=0.8)
    ax[k].set_title(tit)
    ax[k].set_xlabel("frame")
    ax[k].grid(alpha=0.3)
ax[0].legend(fontsize=8, ncol=2)
fig.suptitle(
    "Qualidade da reconstrução na fumaça: leve (tracejado) x alta (contínuo), câmera de teste", fontsize=13
)
fig.tight_layout()
f1 = pasta_fig / "leve_x_alta_metricas_fumaca_por_frame.png"
fig.savefig(f1, dpi=180, bbox_inches="tight")
plt.show()

salvas = [f1.name]
for nome in ordem:
    fig, ax = plt.subplots(3, 4, figsize=(14, 16), facecolor="white")
    for j, f in enumerate(FRAMES_DESTAQUE):
        if (nome, f) not in exemplos:
            continue
        for r, (img, rot) in enumerate(
            zip(exemplos[(nome, f)], ["Real (câmera de teste)", "Reconstrução leve", "Reconstrução alta"])
        ):
            ax[r, j].imshow(np.clip(img, 0, 1))
            ax[r, j].set_xticks([])
            ax[r, j].set_yticks([])
            if r == 0:
                ax[r, j].set_title(f"frame {f}", fontsize=12)
            if j == 0:
                ax[r, j].set_ylabel(rot, fontsize=11)
    fig.suptitle(f"{nome}: foto real x reconstrução leve x reconstrução alta", fontsize=14)
    fig.tight_layout()
    fp = pasta_fig / f"{nome.lower()}_leve_x_alta_frames_040_080_120_160.png"
    fig.savefig(fp, dpi=150, bbox_inches="tight")
    plt.show()
    salvas.append(fp.name)

print("Tabelas em:", saida)
print("Figuras em:", pasta_fig, "|", ", ".join(salvas))


In [ ]:
#@title [Versão alta] Vídeos (MP4 + GIF) da reconstrução 3D dos 3 fluidos + tabela da nuvem de pontos leve x alta
from pathlib import Path
import os, shutil, base64
import numpy as np, pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from PIL import Image
import imageio.v2 as imageio
from IPython.display import HTML, display
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
raiz = Path("/content/drive/MyDrive/TCC2_FluidNexus")
pasta_videos = raiz / "videos"
pasta_videos.mkdir(parents=True, exist_ok=True)
pasta_res = raiz / "resultados" / "comparacao_leve_x_alta"
pasta_res.mkdir(parents=True, exist_ok=True)
INICIO = 20
DESTAQUES = {40, 80, 120, 160}
FLUIDOS = {
    "ScalarReal": ("ScalarReal", "ScalarReal", "train02", 1, 141, None),
    "Smoke": (
        "FluidNexus-Smoke",
        "Smoke",
        "camera02",
        2,
        71,
        raiz / "dados/extraidos/FluidNexusSmoke/camera02/000.png",
    ),
    "Ball": (
        "FluidNexus-Ball",
        "Ball",
        "camera02",
        2,
        71,
        raiz / "dados/extraidos/FluidNexusBall/camera02/000.png",
    ),
}


def rgb(p):
    return np.asarray(Image.open(p).convert("RGB"))


def nuvem(versao_pasta, i):
    for sub in ["fisico", "visual"]:
        p = raiz / "modelos" / versao_pasta / sub / "quantities" / f"frame_{i:03d}_visual_xyz.npy"
        if p.exists():
            return np.load(p)
    return None


linhas = []
for chave, (titulo, pasta, vista, passo, n, _) in FLUIDOS.items():
    for f in sorted(DESTAQUES):
        i = (f - INICIO) // passo
        for versao, sufixo in [("leve", ""), ("alta", "_alta")]:
            pts = nuvem(f"{pasta}_20_160{sufixo}", i)
            if pts is None:
                continue
            ext = np.percentile(pts, 99, axis=0) - np.percentile(pts, 1, axis=0)
            linhas.append(
                {
                    "fluido": chave,
                    "frame": f,
                    "versao": versao,
                    "particulas": len(pts),
                    "largura_x": ext[0],
                    "altura_y": ext[1],
                    "profundidade_z": ext[2],
                }
            )
tab_nuvem = pd.DataFrame(linhas)
tab_nuvem.to_csv(pasta_res / "nuvem_pontos_leve_x_alta_frames_040_080_120_160.csv", index=False)
print("NUVEM DE PONTOS VISUAL: número de partículas por frame (leve x alta)")
display(
    tab_nuvem.pivot_table(index=["fluido", "frame"], columns="versao", values="particulas").reindex(
        list(FLUIDOS), level=0
    )
)

for chave, (titulo, pasta, vista, passo, n, fundo_p) in FLUIDOS.items():
    renders = raiz / f"modelos/{pasta}_20_160_alta/visual/training_render"
    arquivos = sorted(p.name for p in renders.glob(f"*_{vista}*.png"))

    def achar(prefixo, i):
        c = [a for a in arquivos if a.startswith(f"{prefixo}_frame{i:03d}_{vista}")]
        return renders / sorted(c)[-1] if c else None

    indices = [
        i for i in range(n) if achar("real", i) is not None and nuvem(f"{pasta}_20_160_alta", i) is not None
    ]
    print(f"\n{titulo}: {len(indices)} frames", flush=True)

    ult = rgb(achar("real", indices[-1])).astype(np.float32) / 255
    fundo = (
        np.zeros_like(ult)
        if fundo_p is None
        else np.asarray(
            Image.open(fundo_p).convert("RGB").resize((ult.shape[1], ult.shape[0])), dtype=np.float32
        )
        / 255
    )
    ys, xs = np.where(np.abs(ult - fundo).mean(axis=2) > 0.08)
    H, W = ult.shape[:2]
    mg = 60
    y0, y1 = max(0, int(np.percentile(ys, 0.5)) - mg), min(H, int(np.percentile(ys, 99.5)) + mg)
    x0, x1 = max(0, int(np.percentile(xs, 0.5)) - mg), min(W, int(np.percentile(xs, 99.5)) + mg)

    nuvens = {i: nuvem(f"{pasta}_20_160_alta", i) for i in indices}
    todos = np.concatenate([nuvens[indices[0]], nuvens[indices[-1]]])
    lim = [(np.percentile(todos[:, k], 0.5), np.percentile(todos[:, k], 99.5)) for k in range(3)]

    tmp_mp4 = Path(f"/content/{chave.lower()}_reconstrucao_3d_20_160_alta.mp4")
    tmp_gif = Path(f"/content/{chave.lower()}_reconstrucao_3d_20_160_alta.gif")
    escritor = imageio.get_writer(tmp_mp4, fps=6, codec="libx264", quality=8, macro_block_size=8)
    quadros_gif = []
    fig = plt.figure(figsize=(16, 8), dpi=100, facecolor="white")
    for k, i in enumerate(indices):
        frame = INICIO + passo * i
        fig.clf()
        real = rgb(achar("real", i))
        p_rec = achar("render", i)
        rec = rgb(p_rec) if p_rec else np.zeros_like(real)
        if rec.shape != real.shape:
            rec = np.asarray(
                Image.fromarray(rec).resize((real.shape[1], real.shape[0]), Image.Resampling.LANCZOS)
            )
        pts = nuvens[i]
        gs = GridSpec(
            1,
            3,
            figure=fig,
            width_ratios=[1, 1, 1.5],
            left=0.02,
            right=0.99,
            top=0.88,
            bottom=0.05,
            wspace=0.05,
        )
        a1 = fig.add_subplot(gs[0])
        a1.imshow(real[y0:y1, x0:x1])
        a1.set_title("Foto real (câmera de teste)", fontsize=12)
        a1.axis("off")
        a2 = fig.add_subplot(gs[1])
        a2.imshow(rec[y0:y1, x0:x1])
        a2.set_title("Reconstrução renderizada", fontsize=12)
        a2.axis("off")
        a3 = fig.add_subplot(gs[2], projection="3d")
        a3.scatter(
            pts[:, 0],
            pts[:, 2],
            pts[:, 1],
            c=pts[:, 1],
            cmap="viridis",
            s=3,
            alpha=0.6,
            vmin=lim[1][0],
            vmax=lim[1][1],
            linewidths=0,
        )
        a3.set_xlim(*lim[0])
        a3.set_ylim(*lim[2])
        a3.set_zlim(*lim[1])
        a3.set_box_aspect((lim[0][1] - lim[0][0], lim[2][1] - lim[2][0], lim[1][1] - lim[1][0]), zoom=1.3)
        a3.view_init(elev=12, azim=-70 + 360 * k / len(indices))
        a3.set_xticks([])
        a3.set_yticks([])
        a3.set_zticks([])
        a3.set_title(f"Nuvem de pontos 3D ({len(pts):,} partículas)".replace(",", "."), fontsize=12)
        cor = "#c0392b" if frame in DESTAQUES else "black"
        fig.suptitle(f"{titulo}: reconstrução 3D (versão alta)   |   frame {frame}", fontsize=16, color=cor)
        fig.text(0.5, 0.015, "TCC 2 | Felipe Augusto Sousa de Almada", ha="center", fontsize=9, color="gray")
        fig.canvas.draw()
        q = np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy()
        for _ in range(8 if frame in DESTAQUES else 1):
            escritor.append_data(q)
        g = Image.fromarray(q)
        g.thumbnail((900, 900))
        quadros_gif.append(g)
        if k % 20 == 0 or k == len(indices) - 1:
            print(f"  quadro {k + 1}/{len(indices)} (frame {frame})", flush=True)
    escritor.close()
    plt.close(fig)
    quadros_gif[0].save(
        tmp_gif,
        save_all=True,
        append_images=quadros_gif[1:],
        duration=[170] * len(quadros_gif),
        loop=0,
        optimize=True,
    )
    for t in (tmp_mp4, tmp_gif):
        shutil.copy2(t, pasta_videos / t.name)
        print(f"  salvo: {pasta_videos / t.name} ({t.stat().st_size / 1e6:.1f} MB)")
    dados = base64.b64encode(tmp_mp4.read_bytes()).decode()
    display(
        HTML(
            f'<h3>{titulo} (versão alta)</h3><video width="960" controls loop muted><source src="data:video/mp4;base64,{dados}" type="video/mp4"></video>'
        )
    )
print("\nVÍDEOS DA VERSÃO ALTA CONCLUÍDOS")


## 6. Dados do ScalarFlow

In [ ]:
#@title [ScalarFlow] Listar o conteúdo do repositório da TUM (só leitura, não baixa nada)
import os, subprocess

if subprocess.run(["which", "rsync"], capture_output=True).returncode != 0:
    subprocess.run(["apt-get", "install", "-y", "-qq", "rsync"], check=True)
amb = dict(os.environ, RSYNC_PASSWORD="m1521788")
URL = "rsync://m1521788@dataserv.ub.tum.de/m1521788/"


def listar(caminho="", limite=40):
    r = subprocess.run(
        ["rsync", "--list-only", URL + caminho], env=amb, capture_output=True, text=True, timeout=300
    )
    if r.returncode != 0:
        print("ERRO:", r.stderr.strip()[:500])
        return []
    linhas = [l for l in r.stdout.splitlines() if l.strip()]
    print(f"--- {URL}{caminho}  ({len(linhas)} itens)")
    for l in linhas[:limite]:
        print(l)
    if len(linhas) > limite:
        print(f"... (+{len(linhas) - limite} itens)")
    return linhas


raiz = listar()
pastas = [l.split()[-1] for l in raiz if l.startswith("d") and l.split()[-1] not in (".",)]
if pastas:
    p = pastas[0] + "/"
    sub = listar(p)
    for l in sub:
        nome = l.split()[-1]
        if l.startswith("d") and nome not in (".",):
            listar(p + nome + "/", 25)


In [ ]:
#@title [ScalarFlow] Baixar e preparar 10 sequências (densidade 3D completa + velocidade em meia resolução)
import os, io, json, subprocess, tarfile, time, re, builtins

round = builtins.round
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

amb = dict(os.environ, RSYNC_PASSWORD="m1521788")
URL = "rsync://m1521788@dataserv.ub.tum.de/m1521788/"
local = Path("/content/scalarflow_tar")
local.mkdir(exist_ok=True)
destino = Path("/content/drive/MyDrive/TCC2_FluidNexus/dados/scalarflow")
destino.mkdir(parents=True, exist_ok=True)
pasta_fig = Path("/content/drive/MyDrive/TCC2_FluidNexus/figuras")
SEQUENCIAS = [f"sim_{i:06d}" for i in range(10)]


def meia(v):
    x, y, z = (s // 2 * 2 for s in v.shape[:3])
    v = v[:x, :y, :z]
    return v.reshape(x // 2, 2, y // 2, 2, z // 2, 2, -1).mean(axis=(1, 3, 5))


resumo = []
for seq in SEQUENCIAS:
    pronto = destino / seq / "densidade.npz"
    if pronto.exists():
        print(f"{seq}: já preparado antes. Pulando.")
        continue
    t0 = time.time()
    tar = local / f"{seq}.tar"
    if not tar.exists():
        r = subprocess.run(
            ["rsync", "-a", URL + tar.name, str(local)], env=amb, capture_output=True, text=True
        )
        if r.returncode != 0:
            print(f"{seq}: ERRO no download -> {r.stderr[:300]}")
            continue
    t_dl = time.time() - t0
    dens, vel, desc = {}, {}, None
    with tarfile.open(tar) as tf:
        for m in tf:
            if not m.isfile():
                continue
            base = Path(m.name).name
            mm = re.match(r"(density|velocity)_(\d{6})\.npz$", base)
            if mm:
                a = np.load(io.BytesIO(tf.extractfile(m).read()))["data"]
                if mm.group(1) == "density":
                    dens[int(mm.group(2))] = a[..., 0].astype(np.float16)
                else:
                    vel[int(mm.group(2))] = meia(a).astype(np.float16)
            elif base == "description.json":
                desc = tf.extractfile(m).read()
    fd, fv = sorted(dens), sorted(vel)
    D = np.stack([dens[f] for f in fd])
    V = np.stack([vel[f] for f in fv])
    (destino / seq).mkdir(exist_ok=True)
    np.savez_compressed(destino / seq / "velocidade_meia_res.npz", velocidade=V, frames=np.array(fv))
    np.savez_compressed(pronto, densidade=D, frames=np.array(fd))
    if desc:
        (destino / seq / "description.json").write_bytes(desc)
    tar.unlink()
    ocup = 100 * (D.astype(np.float32) > 0.01 * float(D.max())).mean()
    tam = sum(p.stat().st_size for p in (destino / seq).iterdir()) / 1e6
    print(
        f"{seq}: densidade {D.shape}, velocidade {V.shape} | ocupação média {ocup:.1f}% | {tam:.0f} MB no Drive | "
        f"download {t_dl / 60:.1f} min, total {(time.time() - t0) / 60:.1f} min",
        flush=True,
    )
    resumo.append(
        dict(
            sequencia=seq,
            frames=len(fd),
            forma_densidade=str(D.shape[1:]),
            ocupacao_pct=round(ocup, 2),
            mb=round(tam),
        )
    )

seq = SEQUENCIAS[0]
fp = pasta_fig / f"scalarflow_{seq}_densidade_projetada.png"
if not fp.exists() and (destino / seq / "densidade.npz").exists():
    z = np.load(destino / seq / "densidade.npz")
    D, fd = z["densidade"], list(z["frames"])
    mostrar = [f for f in [20, 40, 80, 120, 149] if f in fd]
    fig, ax = plt.subplots(1, len(mostrar), figsize=(3 * len(mostrar), 5), facecolor="white")
    for k, f in enumerate(mostrar):
        img = D[fd.index(f)].astype(np.float32).sum(axis=2).T
        ax[k].imshow(img, origin="lower", cmap="inferno", interpolation="bilinear")
        ax[k].set_title(f"frame {f}")
        ax[k].axis("off")
    fig.suptitle(f"ScalarFlow {seq}: densidade 3D projetada de frente (100x178x100)", fontsize=12)
    fig.tight_layout()
    fig.savefig(fp, dpi=130, bbox_inches="tight")
    plt.show()

if resumo:
    import pandas as pd

    df = pd.DataFrame(resumo)
    display(df)
    csv = destino / "resumo_sequencias.csv"
    df_antigo = pd.read_csv(csv) if csv.exists() else None
    (pd.concat([df_antigo, df]) if df_antigo is not None else df).drop_duplicates(
        "sequencia", keep="last"
    ).to_csv(csv, index=False)
print("SCALARFLOW PREPARADO")


## 7. Segmentação 3D sem ground truth

In [ ]:
%%writefile /content/tcc_seg.py
import os, json, time, math, traceback, warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)
from pathlib import Path
import numpy as np
from scipy import ndimage

GRADE = (64, 112, 64)
BASE = Path(os.environ.get("TCC_BASE", "/content/drive/MyDrive/TCC2_FluidNexus"))
SEG = BASE / "segmentacao"
DADOS, MODELOS, RESULT = SEG / "dados", SEG / "modelos", SEG / "resultados"
FIGS = BASE / "figuras"
SF_SEQS = [f"sim_{i:06d}" for i in range(int(os.environ.get("TCC_N_SF", 10)))]
N_TREINO = int(os.environ.get("TCC_N_TREINO", 6))
N_VAL = int(os.environ.get("TCC_N_VAL", 2))
SPLIT = {
    "treino": SF_SEQS[:N_TREINO],
    "validacao": SF_SEQS[N_TREINO : N_TREINO + N_VAL],
    "teste": SF_SEQS[N_TREINO + N_VAL :],
}
FN_FRAMES = {"ScalarReal": 141, "Smoke": 71, "Ball": 71}
DOMINIO = {
    "ScalarReal": "FluidNexus ScalarReal",
    "Smoke": "FluidNexus Smoke",
    "Ball": "FluidNexus Ball (esparsa)",
}
LIMIAR_CAND = 0.02
MINUTOS = float(os.environ.get("TCC_MINUTOS", 20))
MODELOS_VOL = ["unet3d", "vnet"]
MODELOS_PTS = ["pointnet2", "point_transformer"]
NOMES = {
    "limiar": "Limiar (pseudo-rótulo)",
    "unet3d": "3D U-Net",
    "vnet": "V-Net",
    "pointnet2": "PointNet++",
    "point_transformer": "Point Transformer",
}


def log(*a):
    print(*a, flush=True)


def salvar_json(p, d):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    Path(p).write_text(json.dumps(d, indent=2, ensure_ascii=False))


def normalizar(d):
    pos = d[d > 0]
    esc = float(np.percentile(pos, 99.9)) if pos.size > 100 else float(d.max() or 1.0)
    return np.clip(d / max(esc, 1e-8), 0, 1).astype(np.float32)


def pseudo_rotulo(d):
    """Limiar de Otsu (por sequência) sobre os voxels com fumaça + remoção de partes < 20 voxels."""
    from skimage.filters import threshold_otsu

    vals = d[d > LIMIAR_CAND]
    thr = float(threshold_otsu(vals)) if vals.size > 100 else 0.1
    thr = float(np.clip(thr, 0.05, 0.6))
    m = d > thr
    for t in range(m.shape[0]):
        lab, n = ndimage.label(m[t])
        if n > 1:
            tam = ndimage.sum(np.ones_like(lab), lab, index=np.arange(1, n + 1))
            manter = np.zeros(n + 1, bool)
            manter[1:] = tam >= 20
            m[t] = manter[lab]
    return m, thr


def reamostrar(vol, forma):
    fat = [a / b for a, b in zip(forma, vol.shape[:3])]
    return ndimage.zoom(vol, fat, order=1)


def _salvar_seq(nome, dens, rot, thr, dominio, split, vel=None):
    DADOS.mkdir(parents=True, exist_ok=True)
    extra = {} if vel is None else {"vel": vel.astype(np.float16)}
    np.savez_compressed(
        DADOS / f"{nome}.npz",
        dens=dens.astype(np.float16),
        rot=np.packbits(rot.reshape(-1)),
        forma=np.array(rot.shape),
        **extra,
    )
    salvar_json(
        DADOS / f"{nome}.json",
        {
            "nome": nome,
            "dominio": dominio,
            "split": split,
            "limiar_otsu": thr,
            "frames": int(dens.shape[0]),
            "grade": list(GRADE),
            "ocupacao_rotulo_pct": float(100 * rot.mean()),
        },
    )
    log(
        f"  {nome}: {dens.shape[0]} frames | limiar {thr:.3f} | rótulo ocupa {100 * rot.mean():.2f}% | "
        f'{"com" if vel is not None else "sem"} velocidade'
    )


def carregar_seq(nome, com_vel=False):
    z = np.load(DADOS / f"{nome}.npz")
    forma = tuple(z["forma"])
    rot = np.unpackbits(z["rot"])[: int(np.prod(forma))].reshape(forma).astype(bool)
    meta = json.loads((DADOS / f"{nome}.json").read_text())
    vel = z["vel"] if (com_vel and "vel" in z.files) else None
    return z["dens"], rot, meta, vel


def preparar_scalarflow():
    pasta = BASE / "dados" / "scalarflow"
    for seq in SF_SEQS:
        if (DADOS / f"{seq}.npz").exists():
            log(f"  {seq}: já preparado.")
            continue
        arq = pasta / seq / "densidade.npz"
        if not arq.exists():
            log(f"  {seq}: AUSENTE em {arq} (pulado)")
            continue
        D = np.load(arq)["densidade"].astype(np.float32)
        D = np.stack([reamostrar(d, GRADE) for d in D])
        D = normalizar(D)
        rot, thr = pseudo_rotulo(D)
        split = next(k for k, v in SPLIT.items() if seq in v)
        vel = None
        if split == "teste" and (pasta / seq / "velocidade_meia_res.npz").exists():
            V = np.load(pasta / seq / "velocidade_meia_res.npz")["velocidade"].astype(np.float32)
            fator = GRADE[0] / 100.0
            vel = np.stack(
                [np.stack([reamostrar(v[..., c], GRADE) for c in range(3)], -1) * fator for v in V]
            )
        _salvar_seq(seq, D, rot, thr, "ScalarFlow", split, vel)


def _nuvem(fluido, i):
    for sub in ["fisico", "visual"]:
        p = BASE / "modelos" / f"{fluido}_20_160_alta" / sub / "quantities" / f"frame_{i:03d}_visual_xyz.npy"
        if p.exists():
            return np.load(p)
    return None


def voxelizar(nuvens):
    """Nuvens de pontos (FluidNexus) -> volumes de densidade na grade comum (mesma escala em x, y e z)."""
    amostra = np.concatenate(nuvens[:: max(1, len(nuvens) // 12)])
    lo, hi = np.percentile(amostra, 0.5, axis=0), np.percentile(amostra, 99.5, axis=0)
    centro = (lo + hi) / 2
    esc = max((hi - lo)[k] / (GRADE[k] * 0.9) for k in range(3))
    sobe = np.median(nuvens[-1][:, 1]) - np.median(nuvens[0][:, 1])
    sinal = 1.0 if sobe >= 0 else -1.0
    vols = []
    for p in nuvens:
        q = (p - centro) / esc
        q[:, 1] *= sinal
        idx = np.floor(q + np.array(GRADE) / 2).astype(int)
        ok = np.all((idx >= 0) & (idx < np.array(GRADE)), axis=1)
        v = np.zeros(GRADE, np.float32)
        np.add.at(v, tuple(idx[ok].T), 1.0)
        vols.append(ndimage.gaussian_filter(v, 1.0))
    return np.stack(vols)


def preparar_fluidnexus():
    for fluido, n in FN_FRAMES.items():
        if (DADOS / f"{fluido}.npz").exists():
            log(f"  {fluido}: já preparado.")
            continue
        nuvens = [_nuvem(fluido, i) for i in range(n)]
        nuvens = [p for p in nuvens if p is not None and len(p) > 0]
        if len(nuvens) < 5:
            log(f"  {fluido}: nuvens não encontradas (pulado)")
            continue
        D = normalizar(np.log1p(voxelizar(nuvens)))
        rot, thr = pseudo_rotulo(D)
        _salvar_seq(fluido, D, rot, thr, DOMINIO[fluido], "teste")


def etapa_preparar():
    log("=== ETAPA 1: padronizar dados e gerar pseudo-rótulos ===")
    t0 = time.time()
    preparar_scalarflow()
    preparar_fluidnexus()
    log(f"Etapa 1 concluída em {(time.time() - t0) / 60:.1f} min")


def _torch():
    import torch, torch.nn as nn, torch.nn.functional as F

    return torch, nn, F


def construir_modelo(tipo):
    torch, nn, F = _torch()

    def knn(q, p, k):
        return torch.cdist(q, p).topk(k, dim=-1, largest=False).indices

    def juntar(x, idx):
        b = torch.arange(x.shape[0], device=x.device).view(-1, 1, 1)
        return x[b, idx]

    def fps(p, m):
        B, N, _ = p.shape
        idx = torch.zeros(B, m, dtype=torch.long, device=p.device)
        dist = torch.full((B, N), float("inf"), device=p.device)
        far = torch.randint(0, N, (B,), device=p.device)
        ar = torch.arange(B, device=p.device)
        for i in range(m):
            idx[:, i] = far
            d = ((p - p[ar, far][:, None]) ** 2).sum(-1)
            dist = torch.minimum(dist, d)
            far = dist.argmax(-1)
        return idx

    def interpolar(xyz_d, xyz_s, f_s):
        d = torch.cdist(xyz_d, xyz_s)
        dist, idx = d.topk(min(3, xyz_s.shape[1]), dim=-1, largest=False)
        w = 1.0 / (dist + 1e-8)
        w = w / w.sum(-1, keepdim=True)
        return (juntar(f_s, idx) * w[..., None]).sum(2)

    class MLP(nn.Module):
        def __init__(s, dims):
            super().__init__()
            camadas = []
            for a, b in zip(dims[:-1], dims[1:]):
                camadas += [nn.Linear(a, b), nn.LayerNorm(b), nn.ReLU(inplace=True)]
            s.net = nn.Sequential(*camadas)

        def forward(s, x):
            return s.net(x)

    if tipo == "unet3d":
        from monai.networks.nets import UNet

        return UNet(
            spatial_dims=3,
            in_channels=1,
            out_channels=1,
            channels=(16, 32, 64, 128),
            strides=(2, 2, 2),
            num_res_units=2,
        )
    if tipo == "vnet":
        from monai.networks.nets import VNet

        return VNet(spatial_dims=3, in_channels=1, out_channels=1)

    if tipo == "pointnet2":

        class SA(nn.Module):
            def __init__(s, m, k, dims):
                super().__init__()
                s.m, s.k, s.mlp = m, k, MLP(dims)

            def forward(s, xyz, f):
                i = fps(xyz, s.m)
                nxyz = juntar(xyz, i[..., None])[:, :, 0]
                idx = knn(nxyz, xyz, s.k)
                g = torch.cat([juntar(xyz, idx) - nxyz[:, :, None], juntar(f, idx)], -1)
                return nxyz, s.mlp(g).max(2).values

        class FP(nn.Module):
            def __init__(s, dims):
                super().__init__()
                s.mlp = MLP(dims)

            def forward(s, xyz_d, f_d, xyz_s, f_s):
                return s.mlp(torch.cat([interpolar(xyz_d, xyz_s, f_s), f_d], -1))

        class PointNet2(nn.Module):
            def __init__(s):
                super().__init__()
                s.ent = MLP([4, 32])
                s.sa1 = SA(1024, 24, [3 + 32, 32, 64])
                s.sa2 = SA(256, 24, [3 + 64, 64, 128])
                s.sa3 = SA(64, 24, [3 + 128, 128, 256])
                s.fp3 = FP([256 + 128, 128])
                s.fp2 = FP([128 + 64, 64])
                s.fp1 = FP([64 + 32, 64])
                s.saida = nn.Linear(64, 1)

            def forward(s, x):
                xyz, f0 = x[..., :3], s.ent(x)
                x1, f1 = s.sa1(xyz, f0)
                x2, f2 = s.sa2(x1, f1)
                x3, f3 = s.sa3(x2, f2)
                f2 = s.fp3(x2, f2, x3, f3)
                f1 = s.fp2(x1, f1, x2, f2)
                f0 = s.fp1(xyz, f0, x1, f1)
                return s.saida(f0)[..., 0]

        return PointNet2()

    if tipo == "point_transformer":

        class Bloco(nn.Module):
            def __init__(s, c, k=16):
                super().__init__()
                s.k = k
                s.q, s.kk, s.v = nn.Linear(c, c), nn.Linear(c, c), nn.Linear(c, c)
                s.pos = nn.Sequential(nn.Linear(3, c), nn.ReLU(inplace=True), nn.Linear(c, c))
                s.att = nn.Sequential(nn.Linear(c, c), nn.ReLU(inplace=True), nn.Linear(c, c))
                s.out, s.norm = nn.Linear(c, c), nn.LayerNorm(c)

            def forward(s, xyz, f):
                idx = knn(xyz, xyz, s.k)
                pe = s.pos(xyz[:, :, None] - juntar(xyz, idx))
                w = s.att(s.q(f)[:, :, None] - juntar(s.kk(f), idx) + pe).softmax(dim=2)
                y = (w * (juntar(s.v(f), idx) + pe)).sum(2)
                return s.norm(f + s.out(y))

        class Desce(nn.Module):
            def __init__(s, ci, co, fator=4, k=16):
                super().__init__()
                s.fator, s.k, s.mlp = fator, k, MLP([3 + ci, co])

            def forward(s, xyz, f):
                i = fps(xyz, xyz.shape[1] // s.fator)
                nxyz = juntar(xyz, i[..., None])[:, :, 0]
                idx = knn(nxyz, xyz, s.k)
                g = torch.cat([juntar(xyz, idx) - nxyz[:, :, None], juntar(f, idx)], -1)
                return nxyz, s.mlp(g).max(2).values

        class Sobe(nn.Module):
            def __init__(s, ci, co):
                super().__init__()
                s.a, s.b = MLP([ci, co]), MLP([co, co])

            def forward(s, xyz_d, f_d, xyz_s, f_s):
                return s.b(f_d) + interpolar(xyz_d, xyz_s, s.a(f_s))

        class PointTransformer(nn.Module):
            def __init__(s):
                super().__init__()
                s.ent = MLP([4, 32])
                s.b1 = Bloco(32)
                s.d1 = Desce(32, 64)
                s.b2 = Bloco(64)
                s.d2 = Desce(64, 128)
                s.b3 = Bloco(128)
                s.u2 = Sobe(128, 64)
                s.b4 = Bloco(64)
                s.u1 = Sobe(64, 32)
                s.b5 = Bloco(32)
                s.saida = nn.Linear(32, 1)

            def forward(s, x):
                xyz = x[..., :3]
                f1 = s.b1(xyz, s.ent(x))
                x2, f2 = s.d1(xyz, f1)
                f2 = s.b2(x2, f2)
                x3, f3 = s.d2(x2, f2)
                f3 = s.b3(x3, f3)
                f2 = s.b4(x2, s.u2(x2, f2, x3, f3))
                f1 = s.b5(xyz, s.u1(xyz, f1, x2, f2))
                return s.saida(f1)[..., 0]

        return PointTransformer()
    raise ValueError(tipo)


N_PONTOS = int(os.environ.get("TCC_N_PONTOS", 4096))
_COORD = None


def coords_norm():
    global _COORD
    if _COORD is None:
        g = np.stack(np.meshgrid(*[np.linspace(-1, 1, n) for n in GRADE], indexing="ij"), -1)
        _COORD = g.reshape(-1, 3).astype(np.float32)
    return _COORD


def amostrar_pontos(dens, rot, rng, n=None):
    n = n or N_PONTOS
    d, r = dens.reshape(-1), rot.reshape(-1)
    cand = np.flatnonzero(d > LIMIAR_CAND)
    k = min(len(cand), n // 2)
    sel = np.concatenate(
        [rng.choice(cand, k, replace=False) if k else np.empty(0, int), rng.integers(0, d.size, n - k)]
    )
    x = np.concatenate([coords_norm()[sel], d[sel, None].astype(np.float32)], 1)
    return x, r[sel].astype(np.float32)


def _dados_split(nome_split):
    X, Y = [], []
    for seq in SPLIT[nome_split]:
        if (DADOS / f"{seq}.npz").exists():
            d, r, _, _ = carregar_seq(seq)
            X.append(d)
            Y.append(r)
    if not X:
        raise RuntimeError(f"Nenhuma sequência de {nome_split} preparada.")
    return np.concatenate(X), np.concatenate(Y)


def _dice(p, y, eps=1.0):
    inter = (p * y).sum()
    return float((2 * inter + eps) / (p.sum() + y.sum() + eps))


def treinar(tipo):
    torch, nn, F = _torch()
    MODELOS.mkdir(parents=True, exist_ok=True)
    if (MODELOS / f"{tipo}.json").exists() and (MODELOS / f"{tipo}.pt").exists():
        log(f"{NOMES[tipo]}: já treinado. Pulando.")
        return
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)
    rng = np.random.default_rng(0)
    Xtr, Ytr = _dados_split("treino")
    Xva, Yva = _dados_split("validacao")
    volumetrico = tipo in MODELOS_VOL
    modelo = construir_modelo(tipo).to(dev)
    n_param = sum(p.numel() for p in modelo.parameters())
    opt = torch.optim.AdamW(modelo.parameters(), lr=1e-3, weight_decay=1e-4)
    usar_amp = dev == "cuda" and volumetrico
    scaler = torch.amp.GradScaler("cuda", enabled=usar_amp)
    bce = nn.BCEWithLogitsLoss()
    lote = 4 if volumetrico else 8
    iters = int(os.environ.get("TCC_ITERS", 150))
    if dev == "cuda":
        torch.cuda.reset_peak_memory_stats()
    log(
        f"\n=== Treinando {NOMES[tipo]} | {n_param:,} parâmetros | {len(Xtr)} volumes de treino | "
        f"até {MINUTOS:.0f} min | {dev} ===".replace(",", ".")
    )

    def lote_vol(X, Y, ids, aug):
        x = torch.from_numpy(X[ids].astype(np.float32))[:, None]
        y = torch.from_numpy(Y[ids].astype(np.float32))[:, None]
        if aug:
            if rng.random() < 0.5:
                x, y = x.flip(2), y.flip(2)
            if rng.random() < 0.5:
                x, y = x.flip(4), y.flip(4)
            x = (x * float(rng.uniform(0.8, 1.2)) + 0.03 * torch.randn_like(x)).clamp(0, 1)
        return x.to(dev), y.to(dev)

    def lote_pts(X, Y, ids, aug):
        xs, ys = zip(*[amostrar_pontos(X[i], Y[i], rng) for i in ids])
        x, y = torch.from_numpy(np.stack(xs)), torch.from_numpy(np.stack(ys))
        if aug:
            if rng.random() < 0.5:
                x[..., 0] = -x[..., 0]
            if rng.random() < 0.5:
                x[..., 2] = -x[..., 2]
            x[..., 3] = (x[..., 3] * float(rng.uniform(0.8, 1.2)) + 0.03 * torch.randn_like(x[..., 3])).clamp(
                0, 1
            )
        return x.to(dev), y.to(dev)

    def perda(logit, y):
        p = torch.sigmoid(logit)
        dims = tuple(range(1, p.dim()))
        dice = 1 - (2 * (p * y).sum(dims) + 1) / (p.sum(dims) + y.sum(dims) + 1)
        return bce(logit, y) + dice.mean()

    val_ids = np.arange(0, len(Xva), 3)
    melhor, hist, t0, ep = -1.0, [], time.time(), 0
    while time.time() - t0 < MINUTOS * 60 and ep < int(os.environ.get("TCC_MAX_EPOCAS", 60)):
        ep += 1
        modelo.train()
        soma = 0.0
        for _ in range(iters):
            ids = rng.integers(0, len(Xtr), lote)
            x, y = (lote_vol if volumetrico else lote_pts)(Xtr, Ytr, ids, True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=usar_amp):
                logit = modelo(x)
            l = perda(logit.float(), y)
            scaler.scale(l).backward()
            scaler.step(opt)
            scaler.update()
            soma += float(l)
            if time.time() - t0 > MINUTOS * 60:
                break
        modelo.eval()
        dices = []
        with torch.no_grad():
            for j in range(0, len(val_ids), lote):
                ids = val_ids[j : j + lote]
                x, y = (lote_vol if volumetrico else lote_pts)(Xva, Yva, ids, False)
                with torch.autocast("cuda", enabled=usar_amp):
                    p = (torch.sigmoid(modelo(x).float()) > 0.5).float()
                dices += [_dice(p[i].cpu().numpy(), y[i].cpu().numpy()) for i in range(len(ids))]
        dv = float(np.mean(dices))
        hist.append({"epoca": ep, "perda": soma / iters, "dice_val": dv, "min": (time.time() - t0) / 60})
        log(
            f"  época {ep}: perda {soma / iters:.4f} | Dice validação (vs pseudo-rótulo) {dv:.4f} | "
            f"{(time.time() - t0) / 60:.1f} min"
        )
        if dv > melhor:
            melhor = dv
            torch.save(modelo.state_dict(), MODELOS / f"{tipo}.pt")
    mem = torch.cuda.max_memory_allocated() / 1e9 if dev == "cuda" else 0.0
    salvar_json(
        MODELOS / f"{tipo}.json",
        {
            "modelo": NOMES[tipo],
            "parametros": n_param,
            "epocas": ep,
            "melhor_dice_val": melhor,
            "tempo_treino_min": (time.time() - t0) / 60,
            "memoria_gpu_gb": mem,
            "historico": hist,
        },
    )
    log(
        f"{NOMES[tipo]}: concluído | melhor Dice validação {melhor:.4f} | {(time.time() - t0) / 60:.1f} min | "
        f"memória GPU {mem:.2f} GB"
    )


def etapa_treinar(tipos):
    for t in tipos:
        try:
            treinar(t)
        except Exception:
            log(f"ERRO ao treinar {t}:\n{traceback.format_exc()}")
        finally:
            try:
                import torch

                torch.cuda.empty_cache()
            except Exception:
                pass


def prever(tipo, modelo, dens, dev):
    """Máscara 3D (T, 64,112,64) de um modelo. Só voxels candidatos (densidade > 0,02) podem ser fumaça."""
    torch, _, _ = _torch()
    T = dens.shape[0]
    cand = dens.astype(np.float32) > LIMIAR_CAND
    saida = np.zeros(dens.shape, bool)
    t0 = time.time()
    with torch.no_grad():
        if tipo in MODELOS_VOL:
            for i in range(0, T, 4):
                x = torch.from_numpy(dens[i : i + 4].astype(np.float32))[:, None].to(dev)
                with torch.autocast("cuda", enabled=dev == "cuda"):
                    p = torch.sigmoid(modelo(x).float())[:, 0].cpu().numpy() > 0.5
                saida[i : i + 4] = p
        else:
            rng = np.random.default_rng(0)
            C = coords_norm()
            for t in range(T):
                idx = np.flatnonzero(cand[t].reshape(-1))
                if len(idx) == 0:
                    continue
                idx = rng.permutation(idx)
                pred = np.zeros(len(idx), bool)
                for j in range(0, len(idx), N_PONTOS):
                    parte = idx[j : j + N_PONTOS]
                    pad = np.resize(parte, N_PONTOS)
                    x = np.concatenate([C[pad], dens[t].reshape(-1)[pad, None].astype(np.float32)], 1)
                    lg = modelo(torch.from_numpy(x)[None].to(dev))[0].float().cpu().numpy()
                    pred[j : j + len(parte)] = lg[: len(parte)] > 0
                m = np.zeros(cand[t].size, bool)
                m[idx] = pred
                saida[t] = m.reshape(GRADE)
    ms = 1000 * (time.time() - t0) / T
    return saida & cand, ms


def _iou(a, b):
    u = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / u) if u else np.nan


def _borda(m):
    return m & ~ndimage.binary_erosion(m)


def _hd95(a, b):
    ba, bb = _borda(a), _borda(b)
    if not ba.any() or not bb.any():
        return np.nan
    da = ndimage.distance_transform_edt(~bb)[ba]
    db = ndimage.distance_transform_edt(~ba)[bb]
    return float(max(np.percentile(da, 95), np.percentile(db, 95)))


def _advectar(m, v):
    """Leva a máscara do frame t para t+1 com o campo de velocidade (semi-Lagrangiano, para trás)."""
    g = np.indices(m.shape).astype(np.float32)
    orig = g - np.moveaxis(v.astype(np.float32), -1, 0)
    return ndimage.map_coordinates(m.astype(np.float32), orig, order=1, mode="constant") > 0.5


def metricas(M, rot, vel=None):
    """Métricas sem GT (estabilidade temporal, forma e física) + extras. M e rot: (T, X, Y, Z) bool."""
    T = M.shape[0]
    vol = M.reshape(T, -1).sum(1).astype(float)
    iou_t, var_v, desl, hd, flick, fis, comp = [], [], [], [], [], [], []
    cent = [np.array(ndimage.center_of_mass(M[t])) if vol[t] > 0 else None for t in range(T)]
    for t in range(T):
        lab, n = ndimage.label(M[t])
        comp.append(n)
        if t + 1 < T:
            iou_t.append(_iou(M[t], M[t + 1]))
            if vol[t] > 0:
                var_v.append(abs(vol[t + 1] - vol[t]) / vol[t])
            if cent[t] is not None and cent[t + 1] is not None:
                desl.append(float(np.linalg.norm(cent[t + 1] - cent[t])))
            if t % 2 == 0:
                hd.append(_hd95(M[t], M[t + 1]))
            if vel is not None and t < len(vel):
                fis.append(_iou(_advectar(M[t], vel[t]), M[t + 1]))
        if 0 < t < T - 1:
            troca = (M[t - 1] == M[t + 1]) & (M[t] != M[t - 1])
            flick.append(float(troca.sum() / max(vol[t], 1)))
    fina = rot & ~ndimage.binary_opening(rot, structure=np.ones((1, 3, 3, 3), bool))
    return {
        "consistencia_temporal_iou": float(np.nanmean(iou_t)) if np.isfinite(iou_t).any() else np.nan,
        "variacao_volume": float(np.mean(var_v)) if var_v else np.nan,
        "deslocamento_centroide": float(np.mean(desl)) if desl else np.nan,
        "estabilidade_borda_hd95": float(np.nanmean(hd)) if hd else np.nan,
        "flicker": float(np.mean(flick)) if flick else np.nan,
        "consistencia_fisica_iou": float(np.nanmean(fis)) if np.isfinite(fis).any() else np.nan,
        "frames_vazios_pct": float(100 * (vol == 0).mean()),
        "fracao_volume_pct": float(100 * M.mean()),
        "componentes": float(np.mean(comp)),
        "retencao_estrutura_fina": float((M & fina).sum() / max(fina.sum(), 1)),
        "dice_vs_pseudo": _dice(M.astype(np.float32), rot.astype(np.float32)),
    }


def etapa_avaliar():
    torch, _, _ = _torch()
    import pandas as pd

    RESULT.mkdir(parents=True, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    seqs_teste = [s for s in SPLIT["teste"] if (DADOS / f"{s}.npz").exists()]
    seqs_teste += [f for f in FN_FRAMES if (DADOS / f"{f}.npz").exists()]
    csv = RESULT / "metricas_por_sequencia.csv"
    linhas = pd.read_csv(csv).to_dict("records") if csv.exists() else []
    feitos = {(l["modelo"], l["sequencia"]) for l in linhas}
    exemplos = {}
    for tipo in ["limiar"] + MODELOS_VOL + MODELOS_PTS:
        modelo, info = None, {}
        if tipo != "limiar":
            if not (MODELOS / f"{tipo}.pt").exists():
                log(f"{NOMES[tipo]}: sem modelo treinado (pulado)")
                continue
            try:
                modelo = construir_modelo(tipo).to(dev)
                modelo.load_state_dict(torch.load(MODELOS / f"{tipo}.pt", map_location=dev))
                modelo.eval()
                info = json.loads((MODELOS / f"{tipo}.json").read_text())
            except Exception:
                log(f"ERRO ao carregar {tipo}:\n{traceback.format_exc()}")
                continue
        for seq in seqs_teste:
            if (NOMES[tipo], seq) in feitos:
                continue
            try:
                t0 = time.time()
                dens, rot, meta, vel = carregar_seq(seq, com_vel=True)
                if tipo == "limiar":
                    M, ms = rot, 0.0
                else:
                    M, ms = prever(tipo, modelo, dens, dev)
                met = metricas(M, rot, vel)
                linha = {
                    "modelo": NOMES[tipo],
                    "dominio": meta["dominio"],
                    "sequencia": seq,
                    **met,
                    "inferencia_ms_por_frame": ms,
                    "parametros": info.get("parametros", 0),
                    "tempo_treino_min": info.get("tempo_treino_min", 0.0),
                    "memoria_gpu_gb": info.get("memoria_gpu_gb", 0.0),
                }
                linhas = [l for l in linhas if not (l["modelo"] == NOMES[tipo] and l["sequencia"] == seq)] + [
                    linha
                ]
                pd.DataFrame(linhas).to_csv(csv, index=False)
                t_meio = M.shape[0] // 2
                exemplos[(NOMES[tipo], seq)] = (M[t_meio], dens[t_meio].astype(np.float32))
                log(
                    f'  {NOMES[tipo]} | {seq}: IoU temporal {met["consistencia_temporal_iou"]:.3f} | '
                    f'flicker {met["flicker"]:.3f} | Dice vs pseudo {met["dice_vs_pseudo"]:.3f} | '
                    f"{time.time() - t0:.0f} s"
                )
            except Exception:
                log(f"ERRO avaliando {tipo} em {seq}:\n{traceback.format_exc()}")
        del modelo
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
    if linhas:
        gerar_tabelas_e_figuras(pd.DataFrame(linhas), exemplos)


def gerar_tabelas_e_figuras(df, exemplos):
    import pandas as pd
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    FIGS.mkdir(parents=True, exist_ok=True)
    df = df.copy()
    df["grupo"] = df["dominio"].replace({"ScalarFlow": "ScalarFlow (teste)"})
    cols = [
        "consistencia_temporal_iou",
        "variacao_volume",
        "deslocamento_centroide",
        "estabilidade_borda_hd95",
        "flicker",
        "consistencia_fisica_iou",
        "frames_vazios_pct",
        "fracao_volume_pct",
        "componentes",
        "retencao_estrutura_fina",
        "dice_vs_pseudo",
        "inferencia_ms_por_frame",
    ]
    media = df.groupby(["modelo", "grupo"])[cols].mean().round(4)
    desvio = df.groupby(["modelo", "grupo"])[cols].std().round(4)
    media.to_csv(RESULT / "tabela2_metricas_sem_gt_media.csv")
    desvio.to_csv(RESULT / "tabela2_metricas_sem_gt_desvio.csv")
    custo = df.groupby("modelo")[
        ["parametros", "tempo_treino_min", "memoria_gpu_gb", "inferencia_ms_por_frame"]
    ].mean()
    custo.to_csv(RESULT / "tabela3_custo_computacional.csv")
    log("\nTABELA 2 (média por modelo e domínio) salva em", RESULT / "tabela2_metricas_sem_gt_media.csv")
    try:
        from IPython.display import display

        display(media)
        display(custo.round(2))
    except Exception:
        log(media.to_string())

    ordem_m = [NOMES[k] for k in ["limiar"] + MODELOS_VOL + MODELOS_PTS if NOMES[k] in set(df.modelo)]
    grupos = list(dict.fromkeys(df["grupo"]))
    principais = [
        ("consistencia_temporal_iou", "Consistência temporal (IoU) ↑"),
        ("flicker", "Flicker ↓"),
        ("estabilidade_borda_hd95", "Estabilidade da borda (HD95, voxels) ↓"),
        ("retencao_estrutura_fina", "Retenção de estrutura fina ↑"),
    ]
    fig, ax = plt.subplots(1, 4, figsize=(22, 4.8), facecolor="white")
    w = 0.8 / max(len(ordem_m), 1)
    for k, (c, tit) in enumerate(principais):
        for j, m in enumerate(ordem_m):
            vals = [media.loc[(m, g), c] if (m, g) in media.index else np.nan for g in grupos]
            ax[k].bar(np.arange(len(grupos)) + (j - (len(ordem_m) - 1) / 2) * w, vals, w, label=m)
        ax[k].set_xticks(range(len(grupos)))
        ax[k].set_xticklabels(grupos, rotation=20, ha="right", fontsize=8)
        ax[k].set_title(tit)
        ax[k].grid(axis="y", alpha=0.3)
    ax[0].legend(fontsize=8)
    fig.suptitle("Segmentação 3D sem ground truth: métricas por modelo e domínio", fontsize=13)
    fig.tight_layout()
    fig.savefig(FIGS / "segmentacao_metricas_sem_gt.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    seqs = list(dict.fromkeys(s for (_, s) in exemplos))
    if seqs and ordem_m:
        fig, ax = plt.subplots(
            len(ordem_m) + 1,
            len(seqs),
            figsize=(2.6 * len(seqs), 3.6 * (len(ordem_m) + 1)),
            facecolor="white",
            squeeze=False,
        )
        for c, s in enumerate(seqs):
            dens = next(v[1] for (m, ss), v in exemplos.items() if ss == s)
            ax[0, c].imshow(dens.sum(2).T, origin="lower", cmap="inferno", interpolation="bilinear")
            ax[0, c].set_title(s, fontsize=9)
            for r, m in enumerate(ordem_m, start=1):
                if (m, s) in exemplos:
                    ax[r, c].imshow(exemplos[(m, s)][0].sum(2).T > 0, origin="lower", cmap="gray")
        for r, rot_ in enumerate(["Densidade"] + ordem_m):
            ax[r, 0].set_ylabel(rot_, fontsize=9)
        for a in ax.ravel():
            a.set_xticks([])
            a.set_yticks([])
        fig.suptitle("Máscaras de fumaça (projeção de frente, frame do meio) por modelo", fontsize=13)
        fig.tight_layout()
        fig.savefig(FIGS / "segmentacao_mascaras_por_modelo.png", dpi=130, bbox_inches="tight")
        plt.close(fig)
    log("Figuras salvas em", FIGS)


def executar(etapa, *args):
    """Roda uma etapa sem nunca levantar erro (para a fila de células do Colab não ser cancelada)."""
    t0 = time.time()
    try:
        etapa(*args)
        log(f"OK: {etapa.__name__} em {(time.time() - t0) / 60:.1f} min")
    except Exception:
        log(f"ERRO em {etapa.__name__}:\n{traceback.format_exc()}")


In [ ]:
#@title [Segmentação 1/5] Padronizar dados (grade 64x112x64) e gerar pseudo-rótulos (limiar de Otsu)
import sys, shutil, importlib, subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai"], check=False)
sys.path.insert(0, "/content")
import tcc_seg

importlib.reload(tcc_seg)
destino = tcc_seg.BASE / "codigo" / "segmentacao"
destino.mkdir(parents=True, exist_ok=True)
shutil.copy("/content/tcc_seg.py", destino / "tcc_seg.py")
tcc_seg.executar(tcc_seg.etapa_preparar)


In [ ]:
#@title [Segmentação 2/5] Treinar as redes volumétricas: 3D U-Net e V-Net (até 20 min cada)
import sys, shutil, importlib, subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai"], check=False)
sys.path.insert(0, "/content")
import tcc_seg

importlib.reload(tcc_seg)
tcc_seg.executar(tcc_seg.etapa_treinar, ["unet3d", "vnet"])


In [ ]:
#@title [Segmentação 3/5] Treinar as redes de pontos: PointNet++ e Point Transformer (até 20 min cada)
import sys, shutil, importlib, subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai"], check=False)
sys.path.insert(0, "/content")
import tcc_seg

importlib.reload(tcc_seg)
tcc_seg.executar(tcc_seg.etapa_treinar, ["pointnet2", "point_transformer"])


In [ ]:
#@title [Segmentação 4/5] Avaliar sem ground truth: 6 métricas + extras, tabelas e figuras
import sys, shutil, importlib, subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai"], check=False)
sys.path.insert(0, "/content")
import tcc_seg

importlib.reload(tcc_seg)
tcc_seg.executar(tcc_seg.etapa_avaliar)
from IPython.display import Image, display

for nome in ["segmentacao_metricas_sem_gt.png", "segmentacao_mascaras_por_modelo.png"]:
    p = tcc_seg.FIGS / nome
    if p.exists():
        display(Image(str(p)))


In [ ]:
#@title [Segmentação 5/5] Listar os arquivos de resultado
from pathlib import Path

res = Path("/content/drive/MyDrive/TCC2_FluidNexus/segmentacao/resultados")
print("Arquivos de resultado:", sorted(p.name for p in res.glob("*")) if res.exists() else "nenhum")
